In [1]:
import torch
import time
import pandas as pd
from transformers import AutoModelForCausalLM, AutoTokenizer, DynamicCache
import transformers

transformers.utils.logging.disable_progress_bar()

torch.cuda.set_per_process_memory_fraction(0.45)

# model_name = "gpt2"
model_name = "facebook/opt-6.7b"

device = "cuda"

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    attn_implementation="eager"
).to(device)
model.eval()

print(f"Model: {model_name}")
print(f"Device: {torch.cuda.get_device_name()}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.1f} GB")

Model: facebook/opt-6.7b
Device: NVIDIA A100-SXM4-40GB
VRAM: 39.5 GB


In [2]:
from datasets import load_dataset

# WikiText-2
dataset = load_dataset("wikitext", "wikitext-2-raw-v1", split="test")

# Get long articles by filtering for substantial text
long_prompts = []
for item in dataset:
    text = item["text"].strip()
    tokens = tokenizer.encode(text)
    if len(tokens) > 200:  # only keep long articles
        long_prompts.append(text)
    if len(long_prompts) >= 32:
        break

print(f"Found {len(long_prompts)}  prompts")
for i, p in enumerate(long_prompts[:4]):
    tokens = tokenizer.encode(p)
    print(f"Prompt {i}: {len(tokens)} tokens | {p[:80]}...")

Found 32  prompts
Prompt 0: 217 tokens | He had a recurring role in 2003 on two episodes of The Bill , as character " Con...
Prompt 1: 205 tokens | In 2006 Boulter starred in the play Citizenship written by Mark Ravenhill . The ...
Prompt 2: 246 tokens | Boulter starred in two films in 2008 , Daylight Robbery by filmmaker Paris Leont...
Prompt 3: 268 tokens | Du Fu 's work is notable above all for its range . Chinese critics traditionally...


# **BASELINE - Everything on GPU**

In [3]:
def run_baseline_benchmark():

  microbatches_count = [8,16,32]
  n_tokens = [256,512]
  # microbatches_count = [2]
  # n_tokens = [32]
  results = []

  total_vram_mb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 2)

  for D in microbatches_count:
    for N in n_tokens:

      prompts = long_prompts[:D] # Select D prompts

      torch.cuda.empty_cache()
      torch.cuda.reset_peak_memory_stats()

      total_start= time.perf_counter()
      with torch.no_grad():
        kv_caches = []
        next_tokens = []

        # PREFILL
        for i in range(D):
          inputs = tokenizer(prompts[i], return_tensors="pt").to(device)
          outputs = model(input_ids = inputs.input_ids,use_cache = True)
          kv_caches.append(outputs.past_key_values)
          next_token = torch.argmax(outputs.logits[:,-1,:],dim=1).unsqueeze(1)
          next_tokens.append(next_token)


        torch.cuda.reset_peak_memory_stats()
        # DECODE
        for step in range(N-1):
          for i in range(D):
            outputs = model(
                input_ids = next_tokens[i],
                past_key_values = kv_caches[i],
                use_cache = True
              )
            kv_caches[i] = outputs.past_key_values
            next_tokens[i] = torch.argmax(outputs.logits[:,-1,:],dim=1).unsqueeze(1)

      total_end = time.perf_counter()

      # METRICS
      latency = total_end - total_start
      total_tokens = D * N
      throughput = total_tokens / latency
      peak_vram_mb = torch.cuda.max_memory_allocated() / (1024 ** 2)
      vram_util = (peak_vram_mb / total_vram_mb) * 100

      # KV cache size (sum across all D microbatches)
      total_kv_bytes = 0
      for kv in kv_caches:
        for layer in kv.layers:
          total_kv_bytes += layer.keys.nelement() * layer.keys.element_size()
          total_kv_bytes += layer.values.nelement() * layer.values.element_size()
      kv_cache_mb = total_kv_bytes / (1024 ** 2)

      print(f"D={D:<4} N={N:<4} | Latency={latency:.3f}s | Throughput={throughput:.1f} tok/s | Peak VRAM={peak_vram_mb:.1f}MB | KV Cache={kv_cache_mb:.1f}MB | VRAM%={vram_util:.1f}%")

      results.append({
                "Microbatches (D)": D,
                "Tokens Generated (N)": N,
                "Latency (s)": latency,
                "Throughput (tok/s)": throughput,
                "Peak VRAM (MB)": peak_vram_mb,
                "KV Cache Total (MB)": kv_cache_mb,
                "VRAM Util (%)": vram_util,
            })

      # CLEANUP
      del kv_caches, next_tokens
      torch.cuda.empty_cache()

  return pd.DataFrame(results)

print("Baseline benchmark function ready!")

Baseline benchmark function ready!


In [4]:
run_baseline_benchmark()

D=8    N=256  | Latency=55.842s | Throughput=36.7 tok/s | Peak VRAM=14835.1MB | KV Cache=2053.5MB | VRAM%=36.7%
D=8    N=512  | Latency=110.955s | Throughput=36.9 tok/s | Peak VRAM=16116.6MB | KV Cache=3077.5MB | VRAM%=39.9%
D=16   N=256  | Latency=111.975s | Throughput=36.6 tok/s | Peak VRAM=17314.8MB | KV Cache=4091.5MB | VRAM%=42.8%


OutOfMemoryError: CUDA out of memory. Tried to allocate 20.00 MiB. GPU 0 has a total capacity of 39.49 GiB of which 21.23 GiB is free. Including non-PyTorch memory, this process has 18.25 GiB memory in use. 17.77 GiB allowed; Of the allocated memory 17.14 GiB is allocated by PyTorch, and 628.35 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

# **DEJAVU**

**Helper Functions**

In [2]:
def get_kv_layers(kv_cache):
    """Convert DynamicCache or other cache formats into standard ((k,v),(k,v),...) format."""
    if kv_cache is None:
        return None
    # Newer transformers: DynamicCache stores tensors in .layers[i].keys / .layers[i].values
    if hasattr(kv_cache, "layers") and isinstance(kv_cache.layers, list):
        return [(layer.keys, layer.values) for layer in kv_cache.layers]
    # Older transformers: DynamicCache stores tensors in .key_cache / .value_cache
    if hasattr(kv_cache, "key_cache") and hasattr(kv_cache, "value_cache"):
        return list(zip(kv_cache.key_cache, kv_cache.value_cache))
    # Try legacy tuple conversion as final fallback
    if hasattr(kv_cache, "to_legacy_cache"):
        legacy = kv_cache.to_legacy_cache()
        if isinstance(legacy, (tuple, list)):
            return legacy
    if isinstance(kv_cache, (tuple, list)):
        return kv_cache
    raise TypeError(f"Unsupported KV cache type: {type(kv_cache)}")

def get_cache_seq_len(kv_cache):
    """Return current sequence length of KV cache."""
    layers = get_kv_layers(kv_cache)
    if layers is None or len(layers) == 0:
        return 0
    first_key = layers[0][0]
    return first_key.shape[-2]

def move_full_kv_to_device_timed(kv_cache, target_device):
    """Move KV tensors from CPU → device and reconstruct a DynamicCache the model can consume."""
    start = time.perf_counter()
    dev_layers = [(k.to(target_device), v.to(target_device)) for k, v in kv_cache]
    # Rebuild as DynamicCache so the model receives the correct cache type
    cache = build_dynamic_cache(dev_layers)
    end = time.perf_counter()
    return cache, (end - start)

def extract_new_kv_chunk(full_kv_cache, old_seq_len):
    """Extract only newly generated KV entries from updated cache."""
    layers = get_kv_layers(full_kv_cache)
    new_layers = []
    for k, v in layers:
        new_k = k[:, :, old_seq_len:, :].detach().cpu()
        new_v = v[:, :, old_seq_len:, :].detach().cpu()
        new_layers.append((new_k, new_v))
    return tuple(new_layers)

def build_dynamic_cache(kv_layers):
  """Reconstruct a DynamicCache object from a list of (key, value) tensor pairs."""
  cache = DynamicCache()
  for layer_idx, (k, v) in enumerate(kv_layers):
    # update() expects shape (batch, heads, seq, head_dim) and stores it internally
    cache.update(k, v, layer_idx)
  return cache

def append_kv_caches(cpu_kv_cache, new_chunk_cache):
  """Append new KV chunk to CPU side KV cache."""
  merged = []
  for (old_k, old_v), (new_k, new_v) in zip(cpu_kv_cache, new_chunk_cache):
    merged_k = torch.cat([old_k, new_k], dim=-2)
    merged_v = torch.cat([old_v, new_v], dim=-2)
    merged.append((merged_k, merged_v))
  return tuple(merged)

**BENCHMARK FUNCTION**

In [4]:
import gc
gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()
print(f"Allocated: {torch.cuda.memory_allocated() / (1024**2):.1f} MB")
print(f"Reserved: {torch.cuda.memory_reserved() / (1024**2):.1f} MB")

Allocated: 12700.0 MB
Reserved: 12704.0 MB


In [5]:
def run_dejavu_benchmark():
  microbatches_count = [8,16,32]
  n_tokens = [256,512]
  # microbatches_count = [2]
  # n_tokens = [32]
  results = []

  total_vram_mb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 2)

  for D in microbatches_count:
    for N in n_tokens:

      prompts = long_prompts[:D] # Select D prompts

      torch.cuda.empty_cache()
      torch.cuda.reset_peak_memory_stats()

      total_start= time.perf_counter()

      with torch.no_grad():

        # PREFILL
        cpu_kv_caches = []
        next_tokens = []
        for i in range(D):
            inputs = tokenizer(prompts[i], return_tensors="pt").to(device)
            outputs = model(input_ids=inputs.input_ids, use_cache=True)
            next_tokens.append(torch.argmax(outputs.logits[:, -1, :], dim=1).unsqueeze(1))
            layers = get_kv_layers(outputs.past_key_values)
            cpu_kv_caches.append([(k.detach().cpu(), v.detach().cpu()) for k, v in layers])
            del outputs
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()

        transfer_time = 0.0
        # DECODE
        for step in range(N-1):
          for i in range(D):
            # Transfer KV cache from CPU to GPU
            gpu_cache, t_upload = move_full_kv_to_device_timed(cpu_kv_caches[i], device)
            old_seq_length = get_cache_seq_len(gpu_cache)
            outputs = model(
                input_ids = next_tokens[i],
                past_key_values = gpu_cache,
                use_cache = True
              )
            new_kv_chunk = extract_new_kv_chunk(outputs.past_key_values, old_seq_length)

            # Append newly generated kv chunk to the cpu cache
            t_start = time.perf_counter()
            cpu_kv_caches[i] = append_kv_caches(cpu_kv_caches[i], new_kv_chunk)
            t_stop = time.perf_counter()
            t_offload = t_stop - t_start

            next_tokens[i] = torch.argmax(outputs.logits[:,-1,:],dim=1).unsqueeze(1)

            transfer_time += t_upload
            transfer_time += t_offload

            torch.cuda.empty_cache()

        del gpu_cache, new_kv_chunk
        torch.cuda.empty_cache()
      total_end = time.perf_counter()

      # METRICS
      latency = total_end - total_start
      total_tokens = D * N
      throughput = total_tokens / latency
      peak_vram_mb = torch.cuda.max_memory_allocated() / (1024 ** 2)
      vram_util = (peak_vram_mb / total_vram_mb) * 100
      transfer_overhead = (transfer_time / latency) * 100

      total_kv_bytes = 0
      for cache in cpu_kv_caches:
        for k, v in cache:
          total_kv_bytes += k.nelement() * k.element_size()
          total_kv_bytes += v.nelement() * v.element_size()
      cpu_kv_cache_mb = total_kv_bytes / (1024 ** 2)

      print(f"D={D:<4} N={N:<4} | Latency={latency:.3f}s | Throughput={throughput:.1f} tok/s | Peak VRAM={peak_vram_mb:.1f}MB | VRAM%={vram_util:.1f}% | CPU KV Cache={cpu_kv_cache_mb:.1f}MB | Transfer Time={transfer_time:.2f}s | Transfer Overhead={transfer_overhead:.2f}%")

      results.append({
                "Microbatches (D)": D,
                "Tokens Generated (N)": N,
                "Latency (s)": latency,
                "Throughput (tok/s)": throughput,
                "Peak VRAM (MB)": peak_vram_mb,
                "VRAM Util (%)": vram_util,
                "CPU KV Cache Total (MB)": cpu_kv_cache_mb,
                "Transfer Time (s)": transfer_time,
                "Transfer Overhead (%)": transfer_overhead,
            })

      # CLEANUP
      del next_tokens
      torch.cuda.empty_cache()

  return pd.DataFrame(results)



In [6]:
run_dejavu_benchmark()

D=4    N=128  | Latency=46.834s | Throughput=10.9 tok/s | Peak VRAM=13429.2MB | VRAM%=33.2% | CPU KV Cache=722.0MB | Transfer Time=28.67s | Transfer Overhead=61.21%
D=4    N=256  | Latency=102.338s | Throughput=10.0 tok/s | Peak VRAM=13620.5MB | VRAM%=33.7% | CPU KV Cache=978.0MB | Transfer Time=68.42s | Transfer Overhead=66.85%
D=4    N=512  | Latency=247.317s | Throughput=8.3 tok/s | Peak VRAM=14002.0MB | VRAM%=34.6% | CPU KV Cache=1490.0MB | Transfer Time=178.55s | Transfer Overhead=72.19%
D=8    N=128  | Latency=92.602s | Throughput=11.1 tok/s | Peak VRAM=13954.9MB | VRAM%=34.5% | CPU KV Cache=1541.5MB | Transfer Time=58.80s | Transfer Overhead=63.50%
D=8    N=256  | Latency=208.572s | Throughput=9.8 tok/s | Peak VRAM=13932.5MB | VRAM%=34.5% | CPU KV Cache=2053.5MB | Transfer Time=140.61s | Transfer Overhead=67.41%
D=8    N=512  | Latency=505.531s | Throughput=8.1 tok/s | Peak VRAM=14081.5MB | VRAM%=34.8% | CPU KV Cache=3077.5MB | Transfer Time=368.72s | Transfer Overhead=72.94%
D=

,Microbatches (D),Tokens Generated (N),Latency (s),Throughput (tok/s),Peak VRAM (MB),VRAM Util (%),CPU KV Cache Total (MB),Transfer Time (s),Transfer Overhead (%)
0,4,128,46.834213,10.932179,13429.205078,33.206599,722.0,28.668351,61.212411
1,4,256,102.337944,10.006064,13620.501953,33.679621,978.0,68.415749,66.852768
2,4,512,247.316695,8.280881,14002.017578,34.623001,1490.0,178.548222,72.194165
3,8,128,92.601572,11.058128,13954.894043,34.506478,1541.5,58.803929,63.502085
4,8,256,208.572330,9.819136,13932.542480,34.451209,2053.5,140.605463,67.413287
5,8,512,505.531031,8.102371,14081.518555,34.819584,3077.5,368.717882,72.936746
6,16,128,190.448582,10.753559,15000.038086,37.090821,3067.5,122.573825,64.360587
7,16,256,429.098457,9.545595,15092.327148,37.319026,4091.5,292.347714,68.130684
8,16,512,1046.636200,7.826979,15104.108398,37.348157,6139.5,767.803776,73.359184


# **H2O Baseline - KV Cache Eviction**

In [3]:
def run_h2o_benchmark(budget_ratio=0.4, recent_ratio=0.5):
    """H2O eviction on GPU only — no CPU offloading, no prefetching."""
    microbatches_count = [8, 16,32]
    n_tokens = [256, 512]
    results = []

    total_vram_mb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 2)

    for D in microbatches_count:
        for N in n_tokens:

            prompts = long_prompts[:D]
            torch.cuda.empty_cache()
            torch.cuda.reset_peak_memory_stats()

            sample_len = tokenizer(prompts[0], return_tensors="pt").input_ids.shape[1]
            budget = int((sample_len + N) * budget_ratio)
            budget = max(budget, sample_len)
            recent_window = int(budget * recent_ratio)

            total_start = time.perf_counter()

            with torch.no_grad():

                # PREFILL
                next_tokens = []
                cum_scores_list = []
                kv_caches = []
                for i in range(D):
                    inputs = tokenizer(prompts[i], return_tensors="pt").to(device)
                    outputs = model(input_ids=inputs.input_ids, use_cache=True, output_attentions=True)
                    next_tokens.append(torch.argmax(outputs.logits[:, -1, :], dim=1).unsqueeze(1))

                    seq_len = outputs.past_key_values.get_seq_length()
                    scores = torch.zeros(seq_len, device=device)
                    for attn in outputs.attentions:
                        scores += attn[0].sum(dim=0).sum(dim=0)
                    cum_scores_list.append(scores)
                    kv_caches.append(outputs.past_key_values)
                    # H2O keeps caches on GPU, can't offload here

                torch.cuda.reset_peak_memory_stats()

                # DECODE
                for step in range(N - 1):
                    for i in range(D):

                        outputs = model(
                            input_ids=next_tokens[i],
                            past_key_values=kv_caches[i],
                            use_cache=True,
                            output_attentions=True
                        )
                        kv_caches[i] = outputs.past_key_values
                        next_tokens[i] = torch.argmax(outputs.logits[:, -1, :], dim=1).unsqueeze(1)

                        # Update cumulative scores
                        current_len = kv_caches[i].get_seq_length()
                        new_score = torch.zeros(1, device=device)
                        cum_scores_list[i] = torch.cat([cum_scores_list[i], new_score])
                        for attn in outputs.attentions:
                            step_scores = attn[0].sum(dim=0).sum(dim=0)
                            if len(step_scores) == len(cum_scores_list[i]):
                                cum_scores_list[i] += step_scores

                        # H2O eviction
                        while current_len > budget:
                            evict_candidates = cum_scores_list[i][:-recent_window].clone()
                            if len(evict_candidates) == 0:
                                break

                            evict_pos = evict_candidates.argmin().item()

                            keep_mask = torch.ones(current_len, dtype=torch.bool, device=device)
                            keep_mask[evict_pos] = False
                            keep_indices = torch.where(keep_mask)[0]

                            kv_layers = unpack_kv_cache(kv_caches[i])
                            kept_layers = [(k[:, :, keep_indices, :], v[:, :, keep_indices, :]) for k, v in kv_layers]
                            kv_caches[i] = build_kv_cache(kept_layers)

                            cum_scores_list[i] = cum_scores_list[i][keep_mask]
                            current_len -= 1

                        del outputs

            total_end = time.perf_counter()

            latency = total_end - total_start
            throughput = (D * N) / latency
            peak_vram_mb = torch.cuda.max_memory_allocated() / (1024 ** 2)
            vram_util = (peak_vram_mb / total_vram_mb) * 100

            kv_bytes = 0
            for cache in kv_caches:
                kv_layers = unpack_kv_cache(cache)
                for k, v in kv_layers:
                    kv_bytes += k.nelement() * k.element_size()
                    kv_bytes += v.nelement() * v.element_size()
            kv_mb = kv_bytes / (1024 ** 2)

            print(f"D={D:<4} N={N:<4} | Budget={budget:<4} | Latency={latency:.3f}s | Throughput={throughput:.1f} tok/s | "
                  f"Peak VRAM={peak_vram_mb:.1f}MB | VRAM%={vram_util:.1f}% | GPU KV={kv_mb:.1f}MB")

            results.append({
                "Microbatches (D)": D,
                "Tokens Generated (N)": N,
                "Budget": budget,
                "Latency (s)": latency,
                "Throughput (tok/s)": throughput,
                "Peak VRAM (MB)": peak_vram_mb,
                "VRAM Util (%)": vram_util,
                "GPU KV (MB)": kv_mb,
            })

            del kv_caches, next_tokens, cum_scores_list
            torch.cuda.empty_cache()

    return pd.DataFrame(results)

In [6]:
run_h2o_benchmark()

D=8    N=256  | Budget=217  | Latency=66.340s | Throughput=30.9 tok/s | Peak VRAM=15421.4MB | VRAM%=38.1% | GPU KV=868.0MB
D=8    N=512  | Budget=291  | Latency=129.027s | Throughput=31.7 tok/s | Peak VRAM=15597.1MB | VRAM%=38.6% | GPU KV=1164.0MB
D=16   N=256  | Budget=217  | Latency=131.977s | Throughput=31.0 tok/s | Peak VRAM=16733.3MB | VRAM%=41.4% | GPU KV=1736.0MB
D=16   N=512  | Budget=291  | Latency=258.209s | Throughput=31.7 tok/s | Peak VRAM=16759.5MB | VRAM%=41.4% | GPU KV=2328.0MB


OutOfMemoryError: CUDA out of memory. Tried to allocate 20.00 MiB. GPU 0 has a total capacity of 39.49 GiB of which 21.22 GiB is free. Including non-PyTorch memory, this process has 18.26 GiB memory in use. 17.77 GiB allowed; Of the allocated memory 17.35 GiB is allocated by PyTorch, and 416.27 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

# **DEJAVU+PREFETCH WITHOUT PINNED MEMORY**

In [ ]:
def run_dejavu_parallel_benchmark():
  microbatches_count = [4,8,16]
  n_tokens = [256]
  # microbatches_count = [2]
  # n_tokens = [32]
  results = []

  total_vram_mb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 2)

  for D in microbatches_count:
    for N in n_tokens:

      prompts = LONG_PROMPTS[:D] # Select D prompts

      torch.cuda.empty_cache()
      torch.cuda.reset_peak_memory_stats()

      total_start= time.perf_counter()

      with torch.no_grad():
        kv_caches = []
        next_tokens = []

        # PREFILL
        for i in range(D):
          inputs = tokenizer(prompts[i], return_tensors="pt").to(device)
          outputs = model(input_ids = inputs.input_ids,use_cache = True)
          kv_caches.append(outputs.past_key_values)
          next_token = torch.argmax(outputs.logits[:,-1,:],dim=1).unsqueeze(1)
          next_tokens.append(next_token)

        # Move KV cache to CPU
        cpu_kv_caches = []
        for i in range(D):
          layers = get_kv_layers(kv_caches[i])
          cpu_layers = [(k.detach().cpu(), v.detach().cpu()) for k, v in layers]
          cpu_kv_caches.append(cpu_layers)
        del kv_caches
        torch.cuda.empty_cache()

        transfer_time = 0.0

        prefetch_stream = torch.cuda.Stream()
        offload_stream = torch.cuda.Stream()
        # DECODE

        # Prefetch first microbatch before loop
        with torch.cuda.stream(prefetch_stream):
          prefetched = [(k.to(device, non_blocking=True), v.to(device, non_blocking=True)) for k, v in cpu_kv_caches[0]]

        for step in range(N-1):
          for i in range(D):
            next_i = (i + 1) % D

            prefetch_stream.synchronize()   # make sure upload is done
            offload_stream.synchronize()    # make sure previous offload is done

            # Use current prefetched data
            gpu_layers = prefetched         # this is microbatch i's data

            with torch.cuda.stream(prefetch_stream):
              prefetched = [(k.to(device, non_blocking=True),v.to(device, non_blocking=True)) for k, v in cpu_kv_caches[next_i]]

            gpu_cache = build_dynamic_cache(gpu_layers)
            old_seq_length = gpu_cache.get_seq_length()
            outputs = model(
                input_ids = next_tokens[i],
                past_key_values = gpu_cache,
                use_cache = True
              )

            # Append newly generated kv chunk to the cpu cache
            with torch.cuda.stream(offload_stream):
              new_chunk = extract_new_kv_chunk(outputs.past_key_values, old_seq_length)
              cpu_kv_caches[i] = append_kv_caches(cpu_kv_caches[i], new_chunk)

            next_tokens[i] = torch.argmax(outputs.logits[:,-1,:],dim=1).unsqueeze(1)

        try:
          del gpu_cache, new_kv_chunk
        except:
          pass
        torch.cuda.empty_cache()
      total_end = time.perf_counter()

      # METRICS
      latency = total_end - total_start
      total_tokens = D * N
      throughput = total_tokens / latency
      peak_vram_mb = torch.cuda.max_memory_allocated() / (1024 ** 2)
      vram_util = (peak_vram_mb / total_vram_mb) * 100

      total_kv_bytes = 0
      for cache in cpu_kv_caches:
        for k, v in cache:
          total_kv_bytes += k.nelement() * k.element_size()
          total_kv_bytes += v.nelement() * v.element_size()
      cpu_kv_cache_mb = total_kv_bytes / (1024 ** 2)

      print(f"D={D:<4} N={N:<4} | Latency={latency:.3f}s | Throughput={throughput:.1f} tok/s | Peak VRAM={peak_vram_mb:.1f}MB | VRAM%={vram_util:.1f}% | CPU KV Cache={cpu_kv_cache_mb:.1f}MB")

      results.append({
                "Microbatches (D)": D,
                "Tokens Generated (N)": N,
                "Latency (s)": latency,
                "Throughput (tok/s)": throughput,
                "Peak VRAM (MB)": peak_vram_mb,
                "VRAM Util (%)": vram_util,
                "CPU KV Cache Total (MB)": cpu_kv_cache_mb
            })

      # CLEANUP
      del next_tokens
      torch.cuda.empty_cache()

  return pd.DataFrame(results)



In [ ]:
run_dejavu_parallel_benchmark()

D=4    N=256  | Latency=67.042s | Throughput=15.3 tok/s | Peak VRAM=13631.3MB | VRAM%=33.7% | CPU KV Cache=639.0MB
D=8    N=256  | Latency=132.958s | Throughput=15.4 tok/s | Peak VRAM=13629.4MB | VRAM%=33.7% | CPU KV Cache=1269.0MB
D=16   N=256  | Latency=266.880s | Throughput=15.3 tok/s | Peak VRAM=13809.5MB | VRAM%=34.1% | CPU KV Cache=2537.5MB


,Microbatches (D),Tokens Generated (N),Latency (s),Throughput (tok/s),Peak VRAM (MB),VRAM Util (%),CPU KV Cache Total (MB)
0,4,256,67.042480,15.273898,13631.263184,33.706231,639.0
1,8,256,132.957923,15.403369,13629.381348,33.701578,1269.0
2,16,256,266.880122,15.347715,13809.469727,34.146885,2537.5


# **DEJAVU + PREFETCH**

In [3]:
def unpack_kv_cache(kv_cache):
    """Convert DynamicCache or other cache formats into standard ((k,v),(k,v),...) format."""
    if kv_cache is None:
        return None
    # Newest transformers: DynamicCache has .layers list of DynamicLayer objects
    if hasattr(kv_cache, "layers") and isinstance(kv_cache.layers, list):
        return [(layer.keys, layer.values) for layer in kv_cache.layers]
    # Legacy tuple format
    if isinstance(kv_cache, (tuple, list)):
        return kv_cache
    raise TypeError(f"Unsupported KV cache type: {type(kv_cache)}")


def get_seq_len(kv_cache):
    """Return current sequence length of KV cache."""
    layers = unpack_kv_cache(kv_cache)
    if layers is None or len(layers) == 0:
        return 0
    return layers[0][0].shape[-2]


def kv_cache_size_mb(kv_cache):
    """Compute total KV cache memory size in MB."""
    layers = unpack_kv_cache(kv_cache)
    if layers is None:
        return 0
    total = 0
    for k, v in layers:
        total += k.nelement() * k.element_size()
        total += v.nelement() * v.element_size()
    return total / (1024 ** 2)


def offload_kv_to_cpu_pinned(kv_cache):
    """Move KV cache to pinned CPU memory for async transfer."""
    layers = unpack_kv_cache(kv_cache)
    return tuple(
        (k.detach().contiguous().cpu().pin_memory(),
         v.detach().contiguous().cpu().pin_memory())
        for k, v in layers
    )


def extract_new_kv_chunk_pinned(full_kv_cache, old_seq_len):
    """Extract only newly generated KV entries and store in pinned CPU memory."""
    layers = unpack_kv_cache(full_kv_cache)
    return tuple(
        (k[:, :, old_seq_len:, :].detach().contiguous().cpu().pin_memory(),
         v[:, :, old_seq_len:, :].detach().contiguous().cpu().pin_memory())
        for k, v in layers
    )


def append_kv_chunk_pinned(cpu_kv_cache, new_chunk):
    """Append new KV chunk to CPU-side KV cache, keeping result in pinned memory."""
    return tuple(
        (torch.cat([old_k, new_k], dim=-2).contiguous().pin_memory(),
         torch.cat([old_v, new_v], dim=-2).contiguous().pin_memory())
        for (old_k, old_v), (new_k, new_v) in zip(cpu_kv_cache, new_chunk)
    )


def build_kv_cache(kv_layers):
    """Reconstruct a DynamicCache from a list of (key, value) tensor pairs."""
    cache = DynamicCache()
    for layer_idx, (k, v) in enumerate(kv_layers):
        cache.update(k, v, layer_idx)
    return cache

In [4]:
def run_dejavu_prefetch_benchmark():
    microbatches_count = [8,16,32]
    n_tokens = [256, 512]
    results = []

    total_vram_mb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 2)

    for D in microbatches_count:
        for N in n_tokens:

            prompts = long_prompts[:D]

            torch.cuda.empty_cache()
            torch.cuda.reset_peak_memory_stats()

            total_start = time.perf_counter()

            with torch.no_grad():

                # PREFILL
                cpu_kv_caches = []
                next_tokens = []
                for i in range(D):
                    inputs = tokenizer(prompts[i], return_tensors="pt").to(device)
                    outputs = model(input_ids=inputs.input_ids, use_cache=True)
                    next_tokens.append(torch.argmax(outputs.logits[:, -1, :], dim=1).unsqueeze(1))
                    cpu_kv_caches.append(offload_kv_to_cpu_pinned(outputs.past_key_values))
                    del outputs
                torch.cuda.empty_cache()
                torch.cuda.reset_peak_memory_stats()

                prefetch_stream = torch.cuda.Stream()
                offload_stream = torch.cuda.Stream()

                # Prefetch first microbatch before loop
                with torch.cuda.stream(prefetch_stream):
                    prefetched = [(k.to(device, non_blocking=True), v.to(device, non_blocking=True))
                                  for k, v in cpu_kv_caches[0]]

                # DECODE
                for step in range(N - 1):
                    for i in range(D):
                        next_i = (i + 1) % D

                        prefetch_stream.synchronize()
                        offload_stream.synchronize()

                        gpu_layers = prefetched

                        with torch.cuda.stream(prefetch_stream):
                            prefetched = [(k.to(device, non_blocking=True), v.to(device, non_blocking=True))
                                          for k, v in cpu_kv_caches[next_i]]

                        gpu_cache = build_kv_cache(gpu_layers)
                        old_seq_len = gpu_cache.get_seq_length()
                        outputs = model(
                            input_ids=next_tokens[i],
                            past_key_values=gpu_cache,
                            use_cache=True
                        )

                        with torch.cuda.stream(offload_stream):
                            new_chunk = extract_new_kv_chunk_pinned(outputs.past_key_values, old_seq_len)
                            cpu_kv_caches[i] = append_kv_chunk_pinned(cpu_kv_caches[i], new_chunk)

                        next_tokens[i] = torch.argmax(outputs.logits[:, -1, :], dim=1).unsqueeze(1)

                try:
                    del gpu_cache, new_chunk
                except:
                    pass
                torch.cuda.empty_cache()

            total_end = time.perf_counter()

            latency = total_end - total_start
            throughput = (D * N) / latency
            peak_vram_mb = torch.cuda.max_memory_allocated() / (1024 ** 2)
            vram_util = (peak_vram_mb / total_vram_mb) * 100
            cpu_kv_mb = sum(kv_cache_size_mb(cache) for cache in cpu_kv_caches)

            print(f"D={D:<4} N={N:<4} | Latency={latency:.3f}s | Throughput={throughput:.1f} tok/s | "
                  f"Peak VRAM={peak_vram_mb:.1f}MB | VRAM%={vram_util:.1f}% | CPU KV Cache={cpu_kv_mb:.1f}MB")

            results.append({
                "Microbatches (D)": D,
                "Tokens Generated (N)": N,
                "Latency (s)": latency,
                "Throughput (tok/s)": throughput,
                "Peak VRAM (MB)": peak_vram_mb,
                "VRAM Util (%)": vram_util,
                "CPU KV Cache Total (MB)": cpu_kv_mb
            })

            del next_tokens
            torch.cuda.empty_cache()

    return pd.DataFrame(results)

In [5]:
run_dejavu_prefetch_benchmark()

D=8    N=256  | Latency=114.670s | Throughput=17.9 tok/s | Peak VRAM=13879.3MB | VRAM%=34.3% | CPU KV Cache=2053.5MB
D=8    N=512  | Latency=243.653s | Throughput=16.8 tok/s | Peak VRAM=14373.3MB | VRAM%=35.5% | CPU KV Cache=3077.5MB
D=16   N=256  | Latency=219.008s | Throughput=18.7 tok/s | Peak VRAM=13876.1MB | VRAM%=34.3% | CPU KV Cache=4091.5MB
D=16   N=512  | Latency=493.419s | Throughput=16.6 tok/s | Peak VRAM=14376.4MB | VRAM%=35.5% | CPU KV Cache=6139.5MB
D=32   N=256  | Latency=472.454s | Throughput=17.3 tok/s | Peak VRAM=13980.9MB | VRAM%=34.6% | CPU KV Cache=8323.5MB
D=32   N=512  | Latency=1017.492s | Throughput=16.1 tok/s | Peak VRAM=14487.9MB | VRAM%=35.8% | CPU KV Cache=12419.5MB


,Microbatches (D),Tokens Generated (N),Latency (s),Throughput (tok/s),Peak VRAM (MB),VRAM Util (%),CPU KV Cache Total (MB)
0,8,256,114.670058,17.859937,13879.300781,34.319557,2053.5
1,8,512,243.653211,16.810778,14373.343262,35.541183,3077.5
2,16,256,219.008296,18.702488,13876.113281,34.311675,4091.5
3,16,512,493.418662,16.602534,14376.366699,35.548660,6139.5
4,32,256,472.454237,17.339246,13980.883789,34.570743,8323.5
5,32,512,1017.492024,16.102338,14487.910645,35.824476,12419.5


# **DEJAVU + PREFETCH + H2O**

In [4]:
def run_h2o_dejavu_prefetch_benchmark(budget_ratio=0.4, recent_ratio=0.5):
    microbatches_count = [8,16,32]
    n_tokens = [256, 512]
    results = []

    total_vram_mb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 2)
    num_layers = model.config.num_hidden_layers

    for D in microbatches_count:
        for N in n_tokens:

            prompts = long_prompts[:D]
            torch.cuda.empty_cache()
            torch.cuda.reset_peak_memory_stats()

            # Compute budget per microbatch
            sample_len = tokenizer(prompts[0], return_tensors="pt").input_ids.shape[1]
            budget = int((sample_len + N) * budget_ratio)
            budget = max(budget, sample_len)
            recent_window = int(budget * recent_ratio)

            total_start = time.perf_counter()

            with torch.no_grad():
                next_tokens = []
                cpu_kv_caches = []
                cum_scores_list = []

                # PREFILL
                for i in range(D):
                    inputs = tokenizer(prompts[i], return_tensors="pt").to(device)
                    outputs = model(
                        input_ids=inputs.input_ids,
                        use_cache=True,
                        output_attentions=True
                    )
                    next_token = torch.argmax(outputs.logits[:, -1, :], dim=1).unsqueeze(1)
                    next_tokens.append(next_token)

                    # Initialize cumulative scores
                    seq_len = outputs.past_key_values.get_seq_length()
                    scores = torch.zeros(seq_len, device=device)
                    for attn in outputs.attentions:
                        scores += attn[0].sum(dim=0).sum(dim=0)
                    cum_scores_list.append(scores)

                    # Offload to pinned CPU memory
                    cpu_kv_caches.append(offload_kv_to_cpu_pinned(outputs.past_key_values))

                prefetch_stream = torch.cuda.Stream()
                offload_stream = torch.cuda.Stream()

                # Prefetch first microbatch
                with torch.cuda.stream(prefetch_stream):
                    prefetched = tuple(
                        (k.to(device, non_blocking=True), v.to(device, non_blocking=True))
                        for k, v in cpu_kv_caches[0]
                    )

                # DECODE
                for step in range(N - 1):
                    for i in range(D):
                        next_i = (i + 1) % D

                        prefetch_stream.synchronize()
                        offload_stream.synchronize()

                        gpu_layers = prefetched

                        # Prefetch next microbatch
                        with torch.cuda.stream(prefetch_stream):
                            prefetched = tuple(
                                (k.to(device, non_blocking=True), v.to(device, non_blocking=True))
                                for k, v in cpu_kv_caches[next_i]
                            )

                        # Build cache and decode
                        gpu_cache = build_kv_cache(gpu_layers)
                        outputs = model(
                            input_ids=next_tokens[i],
                            past_key_values=gpu_cache,
                            use_cache=True,
                            output_attentions=True
                        )
                        next_tokens[i] = torch.argmax(outputs.logits[:, -1, :], dim=1).unsqueeze(1)

                        # Update cumulative scores
                        kv = outputs.past_key_values
                        current_len = kv.get_seq_length()
                        new_score = torch.zeros(1, device=device)
                        cum_scores_list[i] = torch.cat([cum_scores_list[i], new_score])
                        for attn in outputs.attentions:
                            step_scores = attn[0].sum(dim=0).sum(dim=0)
                            if len(step_scores) == len(cum_scores_list[i]):
                                cum_scores_list[i] += step_scores

                        # H2O eviction — one at a time
                        while current_len > budget:
                            evict_candidates = cum_scores_list[i][:-recent_window].clone()
                            if len(evict_candidates) == 0:
                                break

                            evict_pos = evict_candidates.argmin().item()

                            keep_mask = torch.ones(current_len, dtype=torch.bool, device=device)
                            keep_mask[evict_pos] = False
                            keep_indices = torch.where(keep_mask)[0]

                            kv_layers = unpack_kv_cache(kv)
                            kept_layers = [(k[:, :, keep_indices, :], v[:, :, keep_indices, :]) for k, v in kv_layers]
                            kv = build_kv_cache(kept_layers)

                            cum_scores_list[i] = cum_scores_list[i][keep_mask]
                            current_len -= 1

                        # Offload evicted cache to pinned CPU
                        with torch.cuda.stream(offload_stream):
                            cpu_kv_caches[i] = tuple(
                                (k.detach().contiguous().cpu().pin_memory(),
                                 v.detach().contiguous().cpu().pin_memory())
                                for k, v in unpack_kv_cache(kv)
                            )

                        del gpu_layers, gpu_cache, outputs, kv

            total_end = time.perf_counter()

            latency = total_end - total_start
            throughput = (D * N) / latency
            peak_vram_mb = torch.cuda.max_memory_allocated() / (1024 ** 2)
            vram_util = (peak_vram_mb / total_vram_mb) * 100

            cpu_kv_mb = sum(
                k.nelement() * k.element_size() + v.nelement() * v.element_size()
                for cache in cpu_kv_caches for k, v in cache
            ) / (1024 ** 2)

            print(f"D={D:<4} N={N:<4} | Budget={budget:<4} | Latency={latency:.3f}s | Throughput={throughput:.1f} tok/s | "
                  f"Peak VRAM={peak_vram_mb:.1f}MB | VRAM%={vram_util:.1f}% | CPU KV={cpu_kv_mb:.1f}MB")

            results.append({
                "Microbatches (D)": D,
                "Tokens Generated (N)": N,
                "Budget": budget,
                "Latency (s)": latency,
                "Throughput (tok/s)": throughput,
                "Peak VRAM (MB)": peak_vram_mb,
                "VRAM Util (%)": vram_util,
                "CPU KV (MB)": cpu_kv_mb,
            })

            del cpu_kv_caches, next_tokens, cum_scores_list, prefetched
            torch.cuda.empty_cache()

    return pd.DataFrame(results)

In [5]:
run_h2o_dejavu_prefetch_benchmark()

D=8    N=256  | Budget=217  | Latency=159.574s | Throughput=12.8 tok/s | Peak VRAM=13782.4MB | VRAM%=34.1% | CPU KV=868.0MB
D=8    N=512  | Budget=291  | Latency=303.474s | Throughput=13.5 tok/s | Peak VRAM=13791.1MB | VRAM%=34.1% | CPU KV=1164.0MB
D=16   N=256  | Budget=217  | Latency=280.293s | Throughput=14.6 tok/s | Peak VRAM=13951.7MB | VRAM%=34.5% | CPU KV=1736.0MB
D=16   N=512  | Budget=291  | Latency=606.618s | Throughput=13.5 tok/s | Peak VRAM=13895.8MB | VRAM%=34.4% | CPU KV=2328.0MB
D=32   N=256  | Budget=217  | Latency=569.757s | Throughput=14.4 tok/s | Peak VRAM=14217.0MB | VRAM%=35.2% | CPU KV=3472.0MB
D=32   N=512  | Budget=291  | Latency=1228.247s | Throughput=13.3 tok/s | Peak VRAM=14136.4MB | VRAM%=35.0% | CPU KV=4656.0MB


,Microbatches (D),Tokens Generated (N),Budget,Latency (s),Throughput (tok/s),Peak VRAM (MB),VRAM Util (%),CPU KV (MB)
0,8,256,217,159.574499,12.834131,13782.368652,34.079872,868.0
1,8,512,291,303.474258,13.497026,13791.126953,34.101528,1164.0
2,16,256,217,280.293100,14.613274,13951.734863,34.498666,1736.0
3,16,512,291,606.617563,13.504390,13895.802246,34.360361,2328.0
4,32,256,217,569.757492,14.378047,14216.997070,35.154584,3472.0
5,32,512,291,1228.247467,13.339331,14136.413086,34.955323,4656.0


# **Profiler** - DejaVu Parallel

In [ ]:
def run_dejavu_profiled(D=16, N=512):
    prompts = LONG_PROMPTS[:D]

    import gc
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    total_vram_mb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 2)

    t_sync_prefetch = 0.0
    t_sync_offload = 0.0
    t_kick_prefetch = 0.0
    t_build_cache = 0.0
    t_forward = 0.0
    t_kick_offload = 0.0
    t_argmax = 0.0

    t_start = time.perf_counter()

    with torch.no_grad():
        kv_caches = []
        next_tokens = []
        for i in range(D):
            inputs = tokenizer(prompts[i], return_tensors="pt").to(device)
            outputs = model(input_ids=inputs.input_ids, use_cache=True)
            kv_caches.append(outputs.past_key_values)
            next_tokens.append(torch.argmax(outputs.logits[:, -1, :], dim=1).unsqueeze(1))

        cpu_kv_caches = []
        for i in range(D):
            layers = unpack_kv_cache(kv_caches[i])
            cpu_layers = tuple((k.detach().cpu(), v.detach().cpu()) for k, v in layers)
            cpu_kv_caches.append(cpu_layers)
        del kv_caches
        torch.cuda.empty_cache()

        prefetch_stream = torch.cuda.Stream()
        offload_stream = torch.cuda.Stream()

        with torch.cuda.stream(prefetch_stream):
            prefetched = tuple((k.to(device, non_blocking=True), v.to(device, non_blocking=True)) for k, v in cpu_kv_caches[0])

        t_decode_start = time.perf_counter()

        with torch.profiler.profile(
            activities=[
                torch.profiler.ProfilerActivity.CPU,
                torch.profiler.ProfilerActivity.CUDA,
            ],
            schedule=torch.profiler.schedule(
                wait=1,
                warmup=1,
                active=3,
                repeat=1
            ),
            on_trace_ready=torch.profiler.tensorboard_trace_handler('./profiler_logs/dejavu_unpinned'),
            record_shapes=True,
            with_stack=False
        ) as prof:
            for step in range(N - 1):
                for i in range(D):
                    next_i = (i + 1) % D

                    t0 = time.perf_counter()
                    prefetch_stream.synchronize()
                    t_sync_prefetch += time.perf_counter() - t0

                    t0 = time.perf_counter()
                    offload_stream.synchronize()
                    t_sync_offload += time.perf_counter() - t0

                    gpu_layers = prefetched

                    t0 = time.perf_counter()
                    with torch.cuda.stream(prefetch_stream):
                        prefetched = tuple((k.to(device, non_blocking=True), v.to(device, non_blocking=True)) for k, v in cpu_kv_caches[next_i])
                    t_kick_prefetch += time.perf_counter() - t0

                    t0 = time.perf_counter()
                    gpu_cache = build_kv_cache(gpu_layers)
                    t_build_cache += time.perf_counter() - t0

                    t0 = time.perf_counter()
                    old_seq = gpu_cache.get_seq_length()
                    with torch.profiler.record_function("forward_pass"):
                        outputs = model(input_ids=next_tokens[i], past_key_values=gpu_cache, use_cache=True)
                    torch.cuda.synchronize()
                    t_forward += time.perf_counter() - t0

                    t0 = time.perf_counter()
                    with torch.profiler.record_function("offload"):
                        with torch.cuda.stream(offload_stream):
                            new_kv = unpack_kv_cache(outputs.past_key_values)
                            new_chunk = tuple((k[:, :, old_seq:, :].detach().cpu(), v[:, :, old_seq:, :].detach().cpu()) for k, v in new_kv)
                            cpu_kv_caches[i] = tuple(
                                (torch.cat([old_k, new_k], dim=-2), torch.cat([old_v, new_v], dim=-2))
                                for (old_k, old_v), (new_k, new_v) in zip(cpu_kv_caches[i], new_chunk)
                            )
                    t_kick_offload += time.perf_counter() - t0

                    t0 = time.perf_counter()
                    next_tokens[i] = torch.argmax(outputs.logits[:, -1, :], dim=1).unsqueeze(1)
                    t_argmax += time.perf_counter() - t0

                    del gpu_layers, gpu_cache, outputs
                    prof.step()

        t_decode_end = time.perf_counter()

    t_end = time.perf_counter()
    decode_time = t_decode_end - t_decode_start
    latency = t_end - t_start
    peak_vram = torch.cuda.max_memory_allocated() / (1024 ** 2)
    cpu_kv_mb = sum(kv_cache_size_mb(cache) for cache in cpu_kv_caches)

    print(f"D={D}, N={N}, Total decode steps: {(N-1)*D}")
    print(f"Decode total:          {decode_time:.3f}s")
    print(f"\n--- Decode breakdown ---")
    print(f"Sync prefetch (wait):  {t_sync_prefetch:.3f}s  ({100*t_sync_prefetch/decode_time:.1f}%)")
    print(f"Sync offload (wait):   {t_sync_offload:.3f}s  ({100*t_sync_offload/decode_time:.1f}%)")
    print(f"Kick prefetch:         {t_kick_prefetch:.3f}s  ({100*t_kick_prefetch/decode_time:.1f}%)")
    print(f"Build cache:           {t_build_cache:.3f}s  ({100*t_build_cache/decode_time:.1f}%)")
    print(f"Forward pass:          {t_forward:.3f}s  ({100*t_forward/decode_time:.1f}%)")
    print(f"Kick offload:          {t_kick_offload:.3f}s  ({100*t_kick_offload/decode_time:.1f}%)")
    print(f"Argmax:                {t_argmax:.3f}s  ({100*t_argmax/decode_time:.1f}%)")

    print("\n--- torch.profiler top ops ---")
    print(prof.key_averages().table(sort_by="cuda_time_total", row_limit=15))

    print(f"\nD={D:<4} N={N:<4} | Latency={latency:.3f}s | Throughput={D*N/latency:.1f} tok/s | Peak VRAM={peak_vram:.1f}MB | VRAM%={100*peak_vram/total_vram_mb:.1f}% | CPU KV={cpu_kv_mb:.1f}MB")

    del cpu_kv_caches, next_tokens, prefetched
    gc.collect()
    torch.cuda.empty_cache()

run_dejavu_profiled()

/usr/local/lib/python3.12/dist-packages/torch/profiler/profiler.py:217: UserWarning: Warning: Profiler clears events at the end of each cycle.Only events from the current cycle will be reported.To keep events across cycles, set acc_events=True.
  _warn_once(


D=16, N=512, Total decode steps: 8176
Decode total:          776.331s

--- Decode breakdown ---
Sync prefetch (wait):  0.176s  (0.0%)
Sync offload (wait):   0.046s  (0.0%)
Kick prefetch:         322.286s  (41.5%)
Build cache:           17.225s  (2.2%)
Forward pass:          295.710s  (38.1%)
Kick offload:          137.675s  (17.7%)
Argmax:                0.853s  (0.1%)

--- torch.profiler top ops ---
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg    # of Calls  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
 

# **Profiler** - DejaVu Parellel with Pinned Memory

In [ ]:
def unpack_kv_cache(kv_cache):
    """Convert DynamicCache or other cache formats into standard ((k,v),(k,v),...) format."""
    if kv_cache is None:
        return None
    # Newest transformers: DynamicCache has .layers list of DynamicLayer objects
    if hasattr(kv_cache, "layers") and isinstance(kv_cache.layers, list):
        return [(layer.keys, layer.values) for layer in kv_cache.layers]
    # Legacy tuple format
    if isinstance(kv_cache, (tuple, list)):
        return kv_cache
    raise TypeError(f"Unsupported KV cache type: {type(kv_cache)}")


def get_seq_len(kv_cache):
    """Return current sequence length of KV cache."""
    layers = unpack_kv_cache(kv_cache)
    if layers is None or len(layers) == 0:
        return 0
    return layers[0][0].shape[-2]


def kv_cache_size_mb(kv_cache):
    """Compute total KV cache memory size in MB."""
    layers = unpack_kv_cache(kv_cache)
    if layers is None:
        return 0
    total = 0
    for k, v in layers:
        total += k.nelement() * k.element_size()
        total += v.nelement() * v.element_size()
    return total / (1024 ** 2)


def offload_kv_to_cpu(kv_cache):
    """Move KV cache to pinned CPU memory for async transfer."""
    layers = unpack_kv_cache(kv_cache)
    return tuple(
        (k.detach().contiguous().cpu().pin_memory(),
         v.detach().contiguous().cpu().pin_memory())
        for k, v in layers
    )


def extract_new_kv_chunk(full_kv_cache, old_seq_len):
    """Extract only newly generated KV entries and store in pinned CPU memory."""
    layers = unpack_kv_cache(full_kv_cache)
    return tuple(
        (k[:, :, old_seq_len:, :].detach().contiguous().cpu().pin_memory(),
         v[:, :, old_seq_len:, :].detach().contiguous().cpu().pin_memory())
        for k, v in layers
    )


def append_kv_chunk(cpu_kv_cache, new_chunk):
    """Append new KV chunk to CPU-side KV cache, keeping result in pinned memory."""
    return tuple(
        (torch.cat([old_k, new_k], dim=-2).contiguous().pin_memory(),
         torch.cat([old_v, new_v], dim=-2).contiguous().pin_memory())
        for (old_k, old_v), (new_k, new_v) in zip(cpu_kv_cache, new_chunk)
    )


def build_kv_cache(kv_layers):
    """Reconstruct a DynamicCache from a list of (key, value) tensor pairs."""
    cache = DynamicCache()
    for layer_idx, (k, v) in enumerate(kv_layers):
        cache.update(k, v, layer_idx)
    return cache

def run_dejavu_benchmark():
    microbatches_count = [16]
    n_tokens = [512]
    results = []

    total_vram_mb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 2)

    for D in microbatches_count:
        for N in n_tokens:

            prompts = LONG_PROMPTS[:D]

            torch.cuda.empty_cache()
            torch.cuda.reset_peak_memory_stats()

            total_start = time.perf_counter()

            # Only profile the D=16, N=512 case
            use_profiler = (D == 16 and N == 512)

            with torch.no_grad():

                # PREFILL
                kv_caches = []
                next_tokens = []
                for i in range(D):
                    inputs = tokenizer(prompts[i], return_tensors="pt").to(device)
                    outputs = model(input_ids=inputs.input_ids, use_cache=True)
                    kv_caches.append(outputs.past_key_values)
                    next_tokens.append(torch.argmax(outputs.logits[:, -1, :], dim=1).unsqueeze(1))

                # Offload prefill KV to pinned CPU memory
                cpu_kv_caches = [offload_kv_to_cpu(kv) for kv in kv_caches]
                del kv_caches
                torch.cuda.empty_cache()

                prefetch_stream = torch.cuda.Stream()
                offload_stream = torch.cuda.Stream()

                # Prefetch first microbatch before loop
                with torch.cuda.stream(prefetch_stream):
                    prefetched = [(k.to(device, non_blocking=True), v.to(device, non_blocking=True))
                                  for k, v in cpu_kv_caches[0]]

                # Timing accumulators
                t_sync_prefetch = 0.0
                t_sync_offload  = 0.0
                t_kick_prefetch = 0.0
                t_build_cache   = 0.0
                t_forward       = 0.0
                t_kick_offload  = 0.0
                t_argmax        = 0.0

                def run_decode_loop(profiler=None):
                    nonlocal prefetched
                    nonlocal t_sync_prefetch, t_sync_offload, t_kick_prefetch
                    nonlocal t_build_cache, t_forward, t_kick_offload, t_argmax

                    for step in range(N - 1):
                        for i in range(D):
                            next_i = (i + 1) % D

                            t0 = time.perf_counter()
                            prefetch_stream.synchronize()
                            t_sync_prefetch += time.perf_counter() - t0

                            t0 = time.perf_counter()
                            offload_stream.synchronize()
                            t_sync_offload += time.perf_counter() - t0

                            gpu_layers = prefetched

                            t0 = time.perf_counter()
                            with torch.cuda.stream(prefetch_stream):
                                prefetched = [(k.to(device, non_blocking=True), v.to(device, non_blocking=True))
                                              for k, v in cpu_kv_caches[next_i]]
                            t_kick_prefetch += time.perf_counter() - t0

                            t0 = time.perf_counter()
                            gpu_cache = build_kv_cache(gpu_layers)
                            old_seq_len = gpu_cache.get_seq_length()
                            t_build_cache += time.perf_counter() - t0

                            t0 = time.perf_counter()
                            with torch.profiler.record_function("forward_pass"):
                                outputs = model(
                                    input_ids=next_tokens[i],
                                    past_key_values=gpu_cache,
                                    use_cache=True
                                )
                            torch.cuda.synchronize()
                            t_forward += time.perf_counter() - t0

                            t0 = time.perf_counter()
                            with torch.profiler.record_function("offload"):
                                with torch.cuda.stream(offload_stream):
                                    new_chunk = extract_new_kv_chunk(outputs.past_key_values, old_seq_len)
                                    cpu_kv_caches[i] = append_kv_chunk(cpu_kv_caches[i], new_chunk)
                            t_kick_offload += time.perf_counter() - t0

                            t0 = time.perf_counter()
                            next_tokens[i] = torch.argmax(outputs.logits[:, -1, :], dim=1).unsqueeze(1)
                            t_argmax += time.perf_counter() - t0

                            if profiler is not None:
                                profiler.step()

                decode_start = time.perf_counter()

                if use_profiler:
                    with torch.profiler.profile(
                        activities=[
                            torch.profiler.ProfilerActivity.CPU,
                            torch.profiler.ProfilerActivity.CUDA,
                        ],
                        schedule=torch.profiler.schedule(
                            wait=1,    # skip first step
                            warmup=1,  # warmup one step
                            active=3,  # record 3 steps
                            repeat=1
                        ),
                        on_trace_ready=torch.profiler.tensorboard_trace_handler('./profiler_logs/dejavu'),
                        record_shapes=True,
                        with_stack=False
                    ) as prof:
                        run_decode_loop(profiler=prof)
                else:
                    run_decode_loop()

                decode_end = time.perf_counter()
                decode_total = decode_end - decode_start
                total_steps = (N - 1) * D

                print(f"\nD={D}, N={N}, Total decode steps: {total_steps}")
                print(f"Decode total:          {decode_total:.3f}s")
                print(f"\n--- Decode breakdown ---")
                print(f"Sync prefetch (wait):  {t_sync_prefetch:.3f}s  ({100*t_sync_prefetch/decode_total:.1f}%)")
                print(f"Sync offload (wait):   {t_sync_offload:.3f}s  ({100*t_sync_offload/decode_total:.1f}%)")
                print(f"Kick prefetch:         {t_kick_prefetch:.3f}s  ({100*t_kick_prefetch/decode_total:.1f}%)")
                print(f"Build cache:           {t_build_cache:.3f}s  ({100*t_build_cache/decode_total:.1f}%)")
                print(f"Forward pass:          {t_forward:.3f}s  ({100*t_forward/decode_total:.1f}%)")
                print(f"Kick offload:          {t_kick_offload:.3f}s  ({100*t_kick_offload/decode_total:.1f}%)")
                print(f"Argmax:                {t_argmax:.3f}s  ({100*t_argmax/decode_total:.1f}%)")

                if use_profiler:
                    print("\n--- torch.profiler top ops ---")
                    print(prof.key_averages().table(
                        sort_by="cuda_time_total",
                        row_limit=15
                    ))

                torch.cuda.empty_cache()

            total_end = time.perf_counter()

            latency = total_end - total_start
            throughput = (D * N) / latency
            peak_vram_mb = torch.cuda.max_memory_allocated() / (1024 ** 2)
            vram_util = (peak_vram_mb / total_vram_mb) * 100
            cpu_kv_mb = sum(kv_cache_size_mb(cache) for cache in cpu_kv_caches)

            print(f"\nD={D:<4} N={N:<4} | Latency={latency:.3f}s | Throughput={throughput:.1f} tok/s | "
                  f"Peak VRAM={peak_vram_mb:.1f}MB | VRAM%={vram_util:.1f}% | CPU KV Cache={cpu_kv_mb:.1f}MB")
            print("-" * 80)

            results.append({
                "Microbatches (D)": D,
                "Tokens Generated (N)": N,
                "Latency (s)": latency,
                "Throughput (tok/s)": throughput,
                "Peak VRAM (MB)": peak_vram_mb,
                "VRAM Util (%)": vram_util,
                "CPU KV Cache Total (MB)": cpu_kv_mb
            })

            del next_tokens, cpu_kv_caches
            torch.cuda.empty_cache()

    return pd.DataFrame(results)

run_dejavu_benchmark()


D=16, N=512, Total decode steps: 8176
Decode total:          519.424s

--- Decode breakdown ---
Sync prefetch (wait):  0.188s  (0.0%)
Sync offload (wait):   0.046s  (0.0%)
Kick prefetch:         10.636s  (2.0%)
Build cache:           17.053s  (3.3%)
Forward pass:          296.844s  (57.1%)
Kick offload:          191.402s  (36.8%)
Argmax:                0.980s  (0.2%)

--- torch.profiler top ops ---
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg    # of Calls  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
  

,Microbatches (D),Tokens Generated (N),Latency (s),Throughput (tok/s),Peak VRAM (MB),VRAM Util (%),CPU KV Cache Total (MB)
0,16,512,523.107525,15.66026,14138.238281,34.959836,4585.5


# **EXTRA**

In [ ]:
def create_kv_buffers(model, D, max_seq_len):
    """Pre-allocate pinned CPU buffers for all microbatches' KV caches."""
    num_layers = model.config.num_hidden_layers
    num_heads = model.config.num_attention_heads
    head_dim = model.config.hidden_size // num_heads
    dtype = next(model.parameters()).dtype

    buffers = []
    current_lens = []

    for i in range(D):
        layer_buffers = []
        for _ in range(num_layers):
            k_buf = torch.zeros(1, num_heads, max_seq_len, head_dim, dtype=dtype).pin_memory()
            v_buf = torch.zeros(1, num_heads, max_seq_len, head_dim, dtype=dtype).pin_memory()
            layer_buffers.append((k_buf, v_buf))
        buffers.append(layer_buffers)
        current_lens.append(0)

    return buffers, current_lens



def unpack_kv_cache(kv_cache):
    """Convert DynamicCache or other cache formats into standard ((k,v),(k,v),...) format."""
    if kv_cache is None:
        return None
    # Newest transformers: DynamicCache has .layers list of DynamicLayer objects
    if hasattr(kv_cache, "layers") and isinstance(kv_cache.layers, list):
        return [(layer.keys, layer.values) for layer in kv_cache.layers]
    # Legacy tuple format
    if isinstance(kv_cache, (tuple, list)):
        return kv_cache
    raise TypeError(f"Unsupported KV cache type: {type(kv_cache)}")


def get_seq_len(kv_cache):
    """Return current sequence length of KV cache."""
    layers = unpack_kv_cache(kv_cache)
    if layers is None or len(layers) == 0:
        return 0
    return layers[0][0].shape[-2]


def kv_cache_size_mb(kv_cache):
    """Compute total KV cache memory size in MB."""
    layers = unpack_kv_cache(kv_cache)
    if layers is None:
        return 0
    total = 0
    for k, v in layers:
        total += k.nelement() * k.element_size()
        total += v.nelement() * v.element_size()
    return total / (1024 ** 2)


def offload_kv_to_cpu(kv_cache):
    """Move KV cache to pinned CPU memory for async transfer."""
    layers = unpack_kv_cache(kv_cache)
    return tuple(
        (k.detach().contiguous().cpu().pin_memory(),
         v.detach().contiguous().cpu().pin_memory())
        for k, v in layers
    )


def extract_new_kv_chunk(full_kv_cache, old_seq_len):
    """Extract only newly generated KV entries and store in pinned CPU memory."""
    layers = unpack_kv_cache(full_kv_cache)
    return tuple(
        (k[:, :, old_seq_len:, :].detach().contiguous().cpu().pin_memory(),
         v[:, :, old_seq_len:, :].detach().contiguous().cpu().pin_memory())
        for k, v in layers
    )


def append_kv_chunk(cpu_kv_cache, new_chunk, step, pin_interval=10):
    should_pin = (step % pin_interval == 0)
    return tuple(
        (torch.cat([old_k, new_k], dim=-2).contiguous().pin_memory() if should_pin else torch.cat([old_k, new_k], dim=-2).contiguous(),
         torch.cat([old_v, new_v], dim=-2).contiguous().pin_memory() if should_pin else torch.cat([old_v, new_v], dim=-2).contiguous())
        for (old_k, old_v), (new_k, new_v) in zip(cpu_kv_cache, new_chunk)
    )


def build_kv_cache(kv_layers):
    """Reconstruct a DynamicCache from a list of (key, value) tensor pairs."""
    cache = DynamicCache()
    for layer_idx, (k, v) in enumerate(kv_layers):
        cache.update(k, v, layer_idx)
    return cache


def run_dejavu_benchmark():
    microbatches_count = [4, 8, 16]
    n_tokens = [128, 256, 512]
    results = []

    total_vram_mb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 2)

    for D in microbatches_count:
        for N in n_tokens:

            prompts = LONG_PROMPTS[:D]

            torch.cuda.empty_cache()
            torch.cuda.reset_peak_memory_stats()

            total_start = time.perf_counter()

            with torch.no_grad():

                sample_len = tokenizer(prompts[0], return_tensors="pt").input_ids.shape[1]
                max_seq = sample_len + N
                buffers, current_lens = create_kv_buffers(model, D, max_seq)

                # PREFILL
                kv_caches = []
                next_tokens = []
                for i in range(D):
                    inputs = tokenizer(prompts[i], return_tensors="pt").to(device)
                    outputs = model(input_ids=inputs.input_ids, use_cache=True)
                    kv_caches.append(outputs.past_key_values)
                    next_tokens.append(torch.argmax(outputs.logits[:, -1, :], dim=1).unsqueeze(1))

                # Offload all KV caches to pinned, Buffered CPU memory
                for i in range(D):
                  kv_layers = unpack_kv_cache(kv_caches[i])
                  seq_len = kv_layers[0][0].shape[2]
                  for j, (k, v) in enumerate(kv_layers):
                      buffers[i][j][0][:, :, :seq_len, :] = k.cpu()
                      buffers[i][j][1][:, :, :seq_len, :] = v.cpu()
                  current_lens[i] = seq_len
                del kv_caches

                prefetch_stream = torch.cuda.Stream()
                offload_stream = torch.cuda.Stream()

                # Prefetch first microbatch before loop
                num_layers = model.config.num_hidden_layers
                with torch.cuda.stream(prefetch_stream):
                    prefetched = [(buffers[0][j][0][:, :, :current_lens[0], :].to(device, non_blocking=True),
                    buffers[0][j][1][:, :, :current_lens[0], :].to(device, non_blocking=True)) for j in range(num_layers)]

                # DECODE
                for step in range(N - 1):
                    for i in range(D):
                        next_i = (i + 1) % D

                        prefetch_stream.synchronize()
                        offload_stream.synchronize()

                        gpu_layers = prefetched

                        with torch.cuda.stream(prefetch_stream):
                            prefetched = [(buffers[next_i][j][0][:, :, :current_lens[next_i], :].to(device, non_blocking=True),
                                           buffers[next_i][j][1][:, :, :current_lens[next_i], :].to(device, non_blocking=True)) for j in range(num_layers)]

                        gpu_cache = build_kv_cache(gpu_layers)
                        old_seq_len = gpu_cache.get_seq_length()
                        outputs = model(
                            input_ids=next_tokens[i],
                            past_key_values=gpu_cache,
                            use_cache=True
                        )

                        with torch.cuda.stream(offload_stream):
                            new_kv = unpack_kv_cache(outputs.past_key_values)
                            for j, (k, v) in enumerate(new_kv):
                                new_k = k[:, :, -1:, :].cpu()
                                new_v = v[:, :, -1:, :].cpu()
                                buffers[i][j][0][:, :, current_lens[i]:current_lens[i]+1, :] = new_k
                                buffers[i][j][1][:, :, current_lens[i]:current_lens[i]+1, :] = new_v
                            current_lens[i] += 1

                        next_tokens[i] = torch.argmax(outputs.logits[:, -1, :], dim=1).unsqueeze(1)

                try:
                    del gpu_cache, new_chunk
                except:
                    pass
                torch.cuda.empty_cache()

            total_end = time.perf_counter()

            latency = total_end - total_start
            throughput = (D * N) / latency
            peak_vram_mb = torch.cuda.max_memory_allocated() / (1024 ** 2)
            vram_util = (peak_vram_mb / total_vram_mb) * 100
            cpu_kv_mb = sum(buffers[i][j][0][:,:,:current_lens[i],:].nelement() * buffers[i][j][0].element_size() * 2
                           for i in range(D) for j in range(num_layers)
                           ) / (1024 ** 2)

            print(f"D={D:<4} N={N:<4} | Latency={latency:.3f}s | Throughput={throughput:.1f} tok/s | "
                  f"Peak VRAM={peak_vram_mb:.1f}MB | VRAM%={vram_util:.1f}% | CPU KV Cache={cpu_kv_mb:.1f}MB")

            results.append({
                "Microbatches (D)": D,
                "Tokens Generated (N)": N,
                "Latency (s)": latency,
                "Throughput (tok/s)": throughput,
                "Peak VRAM (MB)": peak_vram_mb,
                "VRAM Util (%)": vram_util,
                "CPU KV Cache Total (MB)": cpu_kv_mb
            })

            del next_tokens
            torch.cuda.empty_cache()

    return pd.DataFrame(results)

run_dejavu_benchmark()

D=4    N=128  | Latency=28.840s | Throughput=17.8 tok/s | Peak VRAM=13744.7MB | VRAM%=34.0% | CPU KV Cache=371.5MB
D=4    N=256  | Latency=66.600s | Throughput=15.4 tok/s | Peak VRAM=14011.5MB | VRAM%=34.6% | CPU KV Cache=627.5MB
D=4    N=512  | Latency=169.464s | Throughput=12.1 tok/s | Peak VRAM=14512.1MB | VRAM%=35.9% | CPU KV Cache=1139.5MB
D=8    N=128  | Latency=57.366s | Throughput=17.9 tok/s | Peak VRAM=14482.3MB | VRAM%=35.8% | CPU KV Cache=741.5MB
D=8    N=256  | Latency=132.351s | Throughput=15.5 tok/s | Peak VRAM=14010.1MB | VRAM%=34.6% | CPU KV Cache=1253.5MB
D=8    N=512  | Latency=335.447s | Throughput=12.2 tok/s | Peak VRAM=14510.1MB | VRAM%=35.9% | CPU KV Cache=2277.5MB


KeyboardInterrupt: 

In [ ]:
def unpack_kv_cache(kv_cache):
    """Convert DynamicCache or other cache formats into standard ((k,v),(k,v),...) format."""
    if kv_cache is None:
        return None
    # Newest transformers: DynamicCache has .layers list of DynamicLayer objects
    if hasattr(kv_cache, "layers") and isinstance(kv_cache.layers, list):
        return [(layer.keys, layer.values) for layer in kv_cache.layers]
    # Legacy tuple format
    if isinstance(kv_cache, (tuple, list)):
        return kv_cache
    raise TypeError(f"Unsupported KV cache type: {type(kv_cache)}")


def get_seq_len(kv_cache):
    """Return current sequence length of KV cache."""
    layers = unpack_kv_cache(kv_cache)
    if layers is None or len(layers) == 0:
        return 0
    return layers[0][0].shape[-2]


def kv_cache_size_mb(kv_cache):
    """Compute total KV cache memory size in MB."""
    layers = unpack_kv_cache(kv_cache)
    if layers is None:
        return 0
    total = 0
    for k, v in layers:
        total += k.nelement() * k.element_size()
        total += v.nelement() * v.element_size()
    return total / (1024 ** 2)


def offload_kv_to_cpu(kv_cache):
    """Move KV cache to pinned CPU memory for async transfer."""
    layers = unpack_kv_cache(kv_cache)
    return tuple(
        (k.detach().contiguous().cpu().pin_memory(),
         v.detach().contiguous().cpu().pin_memory())
        for k, v in layers
    )


def extract_new_kv_chunk(full_kv_cache, old_seq_len):
    """Extract only newly generated KV entries and store in pinned CPU memory."""
    layers = unpack_kv_cache(full_kv_cache)
    return tuple(
        (k[:, :, old_seq_len:, :].detach().contiguous().cpu().pin_memory(),
         v[:, :, old_seq_len:, :].detach().contiguous().cpu().pin_memory())
        for k, v in layers
    )


def append_kv_chunk(cpu_kv_cache, new_chunk):
    """Append new KV chunk to CPU-side KV cache, keeping result in pinned memory."""
    return tuple(
        (torch.cat([old_k, new_k], dim=-2).contiguous().pin_memory(),
         torch.cat([old_v, new_v], dim=-2).contiguous().pin_memory())
        for (old_k, old_v), (new_k, new_v) in zip(cpu_kv_cache, new_chunk)
    )


def build_kv_cache(kv_layers):
    """Reconstruct a DynamicCache from a list of (key, value) tensor pairs."""
    cache = DynamicCache()
    for layer_idx, (k, v) in enumerate(kv_layers):
        cache.update(k, v, layer_idx)
    return cache

def compute_importance_scores(attentions):
    num_layers = len(attentions)
    num_heads = attentions[0].shape[1]
    num_queries = attentions[0].shape[2]
    num_keys = attentions[0].shape[3]

    scores = torch.zeros(num_keys).to(device)

    for n in range(num_layers):
        layer_attn = attentions[n][0]
        avg_across_heads = layer_attn.sum(dim=0) / num_heads
        avg_across_queries = avg_across_heads.sum(dim=0) / num_queries

        scores += avg_across_queries

    scores = scores / num_layers

    return scores


def classify_indices(top,middle,bottom,scores):
  # sorted scores must be passed here
  sorted_indices = scores.argsort(descending=True)
  top_length = int((len(sorted_indices)/100) * top)
  middle_length = int(((len(sorted_indices)/100)) * middle)
  bottom_lenght = int(((len(sorted_indices)/100)) * bottom)

  hot = sorted_indices[ :top_length]
  cold = sorted_indices[top_length:top_length+middle_length]
  evict = sorted_indices[top_length+middle_length: ]

  return hot,cold,evict

def split_cache(kv_cache, hot_indices, cold_indices):
    hot_layers = []
    cold_layers = []

    for k, v in kv_cache:
        h_idx = hot_indices.to(k.device)
        c_idx = cold_indices.to(k.device)

        hot_k = k[:, :, h_idx, :]
        hot_v = v[:, :, h_idx, :]
        hot_layers.append((hot_k, hot_v))

        cold_k = k[:, :, c_idx, :]
        cold_v = v[:, :, c_idx, :]
        cold_layers.append((cold_k, cold_v))

    return hot_layers, cold_layers

In [ ]:
def run_hotcold_attn_rescore_prefetch_benchmark(hot_pct=20, cold_pct=40, evict_pct=40, rescore_interval=10):
    microbatches_count = [4, 8, 16]
    n_tokens = [128, 256, 512]
    results = []

    total_vram_mb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 2)
    num_layers = model.config.num_hidden_layers

    for D in microbatches_count:
        for N in n_tokens:

            prompts = LONG_PROMPTS[:D]
            torch.cuda.empty_cache()
            torch.cuda.reset_peak_memory_stats()

            total_start = time.perf_counter()

            with torch.no_grad():
                next_tokens = []
                hot_caches = []
                cold_caches = []
                hot_budgets = []
                cum_scores = []

                # PREFILL
                for i in range(D):
                    inputs = tokenizer(prompts[i], return_tensors="pt").to(device)
                    outputs = model(
                        input_ids=inputs.input_ids,
                        use_cache=True,
                        output_attentions=True
                    )
                    next_token = torch.argmax(outputs.logits[:, -1, :], dim=1).unsqueeze(1)
                    next_tokens.append(next_token)

                    scores = compute_importance_scores(outputs.attentions)
                    cum_scores.append(scores)
                    hot_idx, cold_idx, evict_idx = classify_indices(hot_pct, cold_pct, evict_pct, scores)

                    kv_layers = unpack_kv_cache(outputs.past_key_values)
                    hot_layers, cold_layers = split_cache(kv_layers, hot_idx, cold_idx)

                    hot_budgets.append(hot_idx.shape[0])
                    hot_caches.append(hot_layers)
                    cold_caches.append(tuple(
                        (k.detach().contiguous().cpu().pin_memory(),
                         v.detach().contiguous().cpu().pin_memory())
                        for k, v in cold_layers
                    ))

                prefetch_stream = torch.cuda.Stream()
                offload_stream = torch.cuda.Stream()

                # Prefetch first microbatch's cold
                with torch.cuda.stream(prefetch_stream):
                    prefetched = tuple(
                        (k.to(device, non_blocking=True), v.to(device, non_blocking=True))
                        for k, v in cold_caches[0]
                    )

                # DECODE
                for step in range(N - 1):
                    for i in range(D):
                        next_i = (i + 1) % D
                        need_rescore = ((step + 1) % rescore_interval == 0)

                        prefetch_stream.synchronize()
                        offload_stream.synchronize()

                        cold_on_gpu = prefetched

                        # Prefetch next microbatch's cold
                        with torch.cuda.stream(prefetch_stream):
                            prefetched = tuple(
                                (k.to(device, non_blocking=True), v.to(device, non_blocking=True))
                                for k, v in cold_caches[next_i]
                            )

                        # Combine cold + hot
                        combined = []
                        for (ck, cv), (hk, hv) in zip(cold_on_gpu, hot_caches[i]):
                            combined.append((torch.cat([ck, hk], dim=2), torch.cat([cv, hv], dim=2)))
                        gpu_cache = build_kv_cache(combined)

                        # Forward pass
                        outputs = model(
                            input_ids=next_tokens[i],
                            past_key_values=gpu_cache,
                            use_cache=True,
                            output_attentions=need_rescore
                        )
                        next_tokens[i] = torch.argmax(outputs.logits[:, -1, :], dim=1).unsqueeze(1)

                        if need_rescore:
                            # Full reclassification
                            scores = compute_importance_scores(outputs.attentions)
                            cum_scores[i] = cum_scores[i].new_zeros(scores.shape[0])
                            cum_scores[i] += scores

                            hot_idx, cold_idx, evict_idx = classify_indices(hot_pct, cold_pct, evict_pct, cum_scores[i])
                            hot_budgets[i] = hot_idx.shape[0]

                            full_kv = unpack_kv_cache(outputs.past_key_values)
                            hot_layers, cold_layers = split_cache(full_kv, hot_idx, cold_idx)

                            hot_caches[i] = hot_layers

                            with torch.cuda.stream(offload_stream):
                                cold_caches[i] = tuple(
                                    (k.detach().contiguous().cpu().pin_memory(),
                                     v.detach().contiguous().cpu().pin_memory())
                                    for k, v in cold_layers
                                )

                        else:
                            # Recency-based: new token to hot, evict oldest if over budget
                            full_kv = unpack_kv_cache(outputs.past_key_values)
                            new_hot = []
                            for j in range(num_layers):
                                k, v = full_kv[j]
                                new_k = k[:, :, -1:, :]
                                new_v = v[:, :, -1:, :]
                                old_hk, old_hv = hot_caches[i][j]

                                updated_hk = torch.cat([old_hk, new_k], dim=2)
                                updated_hv = torch.cat([old_hv, new_v], dim=2)

                                # Evict oldest if over budget
                                if updated_hk.shape[2] > hot_budgets[i]:
                                    updated_hk = updated_hk[:, :, 1:, :]
                                    updated_hv = updated_hv[:, :, 1:, :]

                                new_hot.append((updated_hk, updated_hv))

                            hot_caches[i] = new_hot

                        del cold_on_gpu, gpu_cache, outputs

              # Calculate before cleanup
                cpu_kv_mb = sum(
                    k.nelement() * k.element_size() + v.nelement() * v.element_size()
                    for cache in cold_caches for k, v in cache
                ) / (1024 ** 2)

                hot_kv_mb = sum(
                    k.nelement() * k.element_size() + v.nelement() * v.element_size()
                    for cache in hot_caches for k, v in cache
                ) / (1024 ** 2)

            total_end = time.perf_counter()

            latency = total_end - total_start
            throughput = (D * N) / latency
            peak_vram_mb = torch.cuda.max_memory_allocated() / (1024 ** 2)
            vram_util = (peak_vram_mb / total_vram_mb) * 100

            print(f"D={D:<4} N={N:<4} | Latency={latency:.3f}s | Throughput={throughput:.1f} tok/s | "
                  f"Peak VRAM={peak_vram_mb:.1f}MB | VRAM%={vram_util:.1f}% | "
                  f"Cold(CPU)={cpu_kv_mb:.1f}MB | Hot(GPU)={hot_kv_mb:.1f}MB")

            results.append({
                "Microbatches (D)": D,
                "Tokens Generated (N)": N,
                "Latency (s)": latency,
                "Throughput (tok/s)": throughput,
                "Peak VRAM (MB)": peak_vram_mb,
                "VRAM Util (%)": vram_util,
                "Cold CPU KV (MB)": cpu_kv_mb,
                "Hot GPU KV (MB)": hot_kv_mb,
            })

            del hot_caches, cold_caches, next_tokens, prefetched, cum_scores
            torch.cuda.empty_cache()

    return pd.DataFrame(results)

In [ ]:
run_hotcold_attn_rescore_prefetch_benchmark()

D=4    N=128  | Latency=16.756s | Throughput=30.6 tok/s | Peak VRAM=12910.0MB | VRAM%=31.9% | Cold(CPU)=0.0MB | Hot(GPU)=0.0MB
D=4    N=256  | Latency=32.551s | Throughput=31.5 tok/s | Peak VRAM=12910.0MB | VRAM%=31.9% | Cold(CPU)=0.0MB | Hot(GPU)=0.0MB
D=4    N=512  | Latency=64.332s | Throughput=31.8 tok/s | Peak VRAM=12910.0MB | VRAM%=31.9% | Cold(CPU)=0.0MB | Hot(GPU)=0.0MB
D=8    N=128  | Latency=33.617s | Throughput=30.5 tok/s | Peak VRAM=12932.0MB | VRAM%=32.0% | Cold(CPU)=0.0MB | Hot(GPU)=0.0MB


KeyboardInterrupt: 

In [ ]:
def validate_hotcold_simple(prompt, max_new_tokens=200, hot_pct=20, cold_pct=50, evict_pct=30, rescore_interval=10):
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    num_layers = model.config.num_hidden_layers

    # BASELINE
    with torch.no_grad():
        kv = DynamicCache()
        outputs = model(input_ids=inputs.input_ids, past_key_values=kv, use_cache=True)
        kv = outputs.past_key_values
        next_token = torch.argmax(outputs.logits[:, -1, :], dim=-1).unsqueeze(1)
        baseline_tokens = [next_token.item()]
        for _ in range(max_new_tokens - 1):
            outputs = model(input_ids=next_token, past_key_values=kv, use_cache=True)
            kv = outputs.past_key_values
            next_token = torch.argmax(outputs.logits[:, -1, :], dim=-1).unsqueeze(1)
            baseline_tokens.append(next_token.item())

    # HOT-COLD
    with torch.no_grad():
        outputs = model(
            input_ids=inputs.input_ids,
            use_cache=True,
            output_attentions=True
        )
        next_token = torch.argmax(outputs.logits[:, -1, :], dim=-1).unsqueeze(1)
        hotcold_tokens = [next_token.item()]

        # Initial classification
        scores = compute_importance_scores(outputs.attentions)
        hot_idx, cold_idx, evict_idx = classify_indices(hot_pct, cold_pct, evict_pct, scores)

        # Keep hot + cold in original order, evict the rest
        kept_idx = torch.cat([hot_idx, cold_idx]).sort().values

        full_kv = unpack_kv_cache(outputs.past_key_values)
        kept_kv = [(k[:, :, kept_idx, :], v[:, :, kept_idx, :]) for k, v in full_kv]

        # Store as single cache on GPU — new tokens append here
        kv = build_kv_cache(kept_kv)

        for step in range(max_new_tokens - 1):
            need_rescore = ((step + 1) % rescore_interval == 0)

            outputs = model(
                input_ids=next_token,
                past_key_values=kv,
                use_cache=True,
                output_attentions=need_rescore
            )
            kv = outputs.past_key_values
            next_token = torch.argmax(outputs.logits[:, -1, :], dim=-1).unsqueeze(1)
            hotcold_tokens.append(next_token.item())

            if need_rescore:
                scores = compute_importance_scores(outputs.attentions)
                hot_idx, cold_idx, evict_idx = classify_indices(hot_pct, cold_pct, evict_pct, scores)
                kept_idx = torch.cat([hot_idx, cold_idx]).sort().values

                full_kv = unpack_kv_cache(kv)
                kept_kv = [(k[:, :, kept_idx, :], v[:, :, kept_idx, :]) for k, v in full_kv]
                kv = build_kv_cache(kept_kv)

            del outputs

    total = len(baseline_tokens)
    match = sum(a == b for a, b in zip(baseline_tokens, hotcold_tokens))
    print(f"  {match}/{total} ({100*match/total:.1f}%) | {tokenizer.decode(hotcold_tokens[:20])}")

for p in LONG_PROMPTS[:4]:
    print(f"\nPrompt: {p[:60]}...")
    print("  Baseline: ", tokenizer.decode([
        torch.argmax(model(input_ids=tokenizer(p, return_tensors="pt").to(device).input_ids, use_cache=True).logits[:, -1, :]).item()
    ]))
    for interval in [5, 10, 20, 50]:
        print(f"  Rescore every {interval}:", end="")
        validate_hotcold_simple(p, rescore_interval=interval)


Prompt: Artificial intelligence has transformed the way we approach ...
  Baseline:   Artificial
  Rescore every 5:  8/200 (4.0%) |  Artificial intelligence has transformed the way we approach the world.

The world is a complex place.
  Rescore every 10:  23/200 (11.5%) |  Artificial intelligence has transformed the way we approach complex problems in science and engineering Artificial intelligence has transformed the way
  Rescore every 20:  200/200 (100.0%) |  Artificial intelligence has transformed the way we approach complex problems in science and engineering Artificial intelligence has transformed the way
  Rescore every 50:  200/200 (100.0%) |  Artificial intelligence has transformed the way we approach complex problems in science and engineering Artificial intelligence has transformed the way

Prompt: The history of ancient Rome spans over a thousand years from...
  Baseline:   The
  Rescore every 5:  19/200 (9.5%) |  The history of ancient Rome spans over a period of time fro

In [ ]:
long_test = """The transformer architecture was introduced in 2017 by Vaswani et al. in their paper titled Attention Is All You Need. The architecture relies entirely on self-attention mechanisms, dispensing with recurrence and convolutions entirely. Since its introduction, the transformer has become the foundation for nearly all modern large language models. Models like GPT, BERT, T5, and LLaMA all build upon the original transformer design. The key innovation was the multi-head self-attention mechanism, which allows the model to attend to different positions in the input sequence simultaneously. This parallel attention computation made transformers significantly more efficient to train than recurrent neural networks, which process tokens sequentially. The transformer uses positional encodings to inject information about the order of tokens in the sequence, since the self-attention mechanism itself is permutation invariant. The original transformer was designed for machine translation, but researchers quickly discovered that the architecture could be adapted for a wide variety of natural language processing tasks."""

for interval in [10, 20, 50]:
    print(f"Rescore every {interval}:", end="")
    validate_hotcold_simple(long_test, max_new_tokens=200, hot_pct=20, cold_pct=50, evict_pct=30, rescore_interval=interval)

Rescore every 10:  7/200 (3.5%) | 

The transformer architecture was introduced in 2017 by Vaswani et al. in their paper titled
Rescore every 20:  5/200 (2.5%) | 

The transformer architecture was introduced in 2017 by Vaswani et al. in their paper titled
Rescore every 50:  5/200 (2.5%) | 

The transformer architecture was introduced in 2017 by Vaswani et al. in their paper titled


# **H2O VALIDATION**

In [ ]:
def validate_h2o_proper(prompt, max_new_tokens=200, budget_ratio=0.8, recent_ratio=0.5):
    """
    Proper H2O implementation:
    - Fixed cache budget = prompt_len * budget_ratio
    - Budget split: half for recent tokens, half for heavy hitters
    - Evict ONE lowest-scoring non-recent token when cache exceeds budget
    - Cumulative attention scores updated every step
    """
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    prompt_len = inputs.input_ids.shape[1]
    budget = int((prompt_len + max_new_tokens) * budget_ratio)
    recent_window = int(budget * recent_ratio)  # half budget for recent
    num_layers = model.config.num_hidden_layers

    # BASELINE
    with torch.no_grad():
        kv = DynamicCache()
        outputs = model(input_ids=inputs.input_ids, past_key_values=kv, use_cache=True)
        kv = outputs.past_key_values
        next_token = torch.argmax(outputs.logits[:, -1, :], dim=-1).unsqueeze(1)
        baseline_tokens = [next_token.item()]
        for _ in range(max_new_tokens - 1):
            outputs = model(input_ids=next_token, past_key_values=kv, use_cache=True)
            kv = outputs.past_key_values
            next_token = torch.argmax(outputs.logits[:, -1, :], dim=-1).unsqueeze(1)
            baseline_tokens.append(next_token.item())

    # H2O
    with torch.no_grad():
        outputs = model(
            input_ids=inputs.input_ids,
            use_cache=True,
            output_attentions=True
        )
        next_token = torch.argmax(outputs.logits[:, -1, :], dim=-1).unsqueeze(1)
        h2o_tokens = [next_token.item()]

        # Initialize cumulative scores from prefill
        seq_len = outputs.past_key_values.get_seq_length()
        cum_scores = torch.zeros(seq_len, device=device)
        for attn in outputs.attentions:
            cum_scores += attn[0].sum(dim=0).sum(dim=0)

        kv = outputs.past_key_values

        for step in range(max_new_tokens - 1):
            outputs = model(
                input_ids=next_token,
                past_key_values=kv,
                use_cache=True,
                output_attentions=True
            )
            kv = outputs.past_key_values
            next_token = torch.argmax(outputs.logits[:, -1, :], dim=-1).unsqueeze(1)
            h2o_tokens.append(next_token.item())

            # Update cumulative scores — new token starts at 0
            current_len = kv.get_seq_length()
            new_score = torch.zeros(1, device=device)
            cum_scores = torch.cat([cum_scores, new_score])

            # Add this step's attention scores
            for attn in outputs.attentions:
                step_scores = attn[0].sum(dim=0).sum(dim=0)
                if len(step_scores) == len(cum_scores):
                    cum_scores += step_scores

            # Evict if over budget — one at a time
            while current_len > budget:
                # Protect recent tokens — last recent_window positions
                evict_candidates = cum_scores[:-recent_window].clone()

                if len(evict_candidates) == 0:
                    break

                # Find lowest scoring non-recent token
                evict_pos = evict_candidates.argmin().item()

                # Remove from cache
                keep_mask = torch.ones(current_len, dtype=torch.bool, device=device)
                keep_mask[evict_pos] = False
                keep_indices = torch.where(keep_mask)[0]

                kv_layers = unpack_kv_cache(kv)
                kept_layers = [(k[:, :, keep_indices, :], v[:, :, keep_indices, :]) for k, v in kv_layers]
                kv = build_kv_cache(kept_layers)

                # Remove from scores
                cum_scores = cum_scores[keep_mask]
                current_len -= 1

            del outputs

    total = len(baseline_tokens)
    match = sum(a == b for a, b in zip(baseline_tokens, h2o_tokens))
    print(f"  Budget={budget} (ratio={budget_ratio}) | Recent window={recent_window}")
    print(f"  Match: {match}/{total} ({100*match/total:.1f}%)")
    print(f"  Baseline: {tokenizer.decode(baseline_tokens[:25])}")
    print(f"  H2O:      {tokenizer.decode(h2o_tokens[:25])}")

# Test with different budget ratios
for p in LONG_PROMPTS[:4]:
    print(f"\nPrompt: {p[:60]}...")
    validate_h2o_proper(p, budget_ratio=0.9)
    validate_h2o_proper(p, budget_ratio=0.8)
    validate_h2o_proper(p, budget_ratio=0.7)


Prompt: Artificial intelligence has transformed the way we approach ...
  Budget=232 (ratio=0.9) | Recent window=116
  Match: 200/200 (100.0%)
  Baseline:  Artificial intelligence has transformed the way we approach complex problems in science and engineering Artificial intelligence has transformed the way we approach complex problems in
  H2O:       Artificial intelligence has transformed the way we approach complex problems in science and engineering Artificial intelligence has transformed the way we approach complex problems in
  Budget=206 (ratio=0.8) | Recent window=103
  Match: 200/200 (100.0%)
  Baseline:  Artificial intelligence has transformed the way we approach complex problems in science and engineering Artificial intelligence has transformed the way we approach complex problems in
  H2O:       Artificial intelligence has transformed the way we approach complex problems in science and engineering Artificial intelligence has transformed the way we approach complex problems 

In [ ]:
for p in LONG_PROMPTS[:4]:
    print(f"\nPrompt: {p[:60]}...")
    validate_h2o_proper(p, budget_ratio=0.5)
    validate_h2o_proper(p, budget_ratio=0.4)
    validate_h2o_proper(p, budget_ratio=0.3)


Prompt: Artificial intelligence has transformed the way we approach ...
  Budget=129 (ratio=0.5) | Recent window=64
  Match: 200/200 (100.0%)
  Baseline:  Artificial intelligence has transformed the way we approach complex problems in science and engineering Artificial intelligence has transformed the way we approach complex problems in
  H2O:       Artificial intelligence has transformed the way we approach complex problems in science and engineering Artificial intelligence has transformed the way we approach complex problems in
  Budget=103 (ratio=0.4) | Recent window=51
  Match: 200/200 (100.0%)
  Baseline:  Artificial intelligence has transformed the way we approach complex problems in science and engineering Artificial intelligence has transformed the way we approach complex problems in
  H2O:       Artificial intelligence has transformed the way we approach complex problems in science and engineering Artificial intelligence has transformed the way we approach complex problems in

In [ ]:
for p in LONG_PROMPTS[:4]:
    print(f"\nPrompt: {p[:60]}...")
    validate_h2o_proper(p, budget_ratio=0.2)


Prompt: Artificial intelligence has transformed the way we approach ...
  Budget=51 (ratio=0.2) | Recent window=25
  Match: 200/200 (100.0%)
  Baseline:  Artificial intelligence has transformed the way we approach complex problems in science and engineering Artificial intelligence has transformed the way we approach complex problems in
  H2O:       Artificial intelligence has transformed the way we approach complex problems in science and engineering Artificial intelligence has transformed the way we approach complex problems in

Prompt: The history of ancient Rome spans over a thousand years from...
  Budget=53 (ratio=0.2) | Recent window=26
  Match: 17/200 (8.5%)
  Baseline:  The history of ancient Rome spans over a thousand years from its founding as a small village The history of ancient Rome spans over a
  H2O:       The history of ancient Rome spans over a thousand years from its founding as a village The history of ancient Rome spans over a thousand

Prompt: Quantum computing l

In [ ]:
long_unique = """Large language models have revolutionized natural language processing by demonstrating remarkable capabilities in text generation, translation, summarization, and question answering. These models, built on the transformer architecture introduced by Vaswani et al in 2017, use self-attention mechanisms to capture long-range dependencies in text. The key innovation of transformers is the multi-head attention mechanism, which allows the model to attend to information from different representation subspaces at different positions. During inference, the model generates tokens autoregressively, producing one token at a time based on the entire previous context. This requires storing key and value tensors from all previous tokens in what is known as the KV cache. As sequence lengths grow, the KV cache becomes a significant memory bottleneck, consuming gigabytes of GPU memory for long contexts. Several approaches have been proposed to address this challenge, including quantization of cached values, eviction of less important entries, and offloading to CPU memory."""

validate_h2o_proper(long_unique, max_new_tokens=200, budget_ratio=0.5)
validate_h2o_proper(long_unique, max_new_tokens=200, budget_ratio=0.3)
validate_h2o_proper(long_unique, max_new_tokens=200, budget_ratio=0.2)

  Budget=197 (ratio=0.5) | Recent window=98
  Match: 12/200 (6.0%)
  Baseline:  In this paper, we propose a novel approach to address the memory bottleneck of large language models. We propose a novel approach to
  H2O:       In this paper, we propose a novel approach to address this problem, called the KV-cache-free model. The
  Budget=118 (ratio=0.3) | Recent window=59
  Match: 13/200 (6.5%)
  Baseline:  In this paper, we propose a novel approach to address the memory bottleneck of large language models. We propose a novel approach to
  H2O:       In this paper, we propose a novel approach to offload the memory burden of large language models. We use a novel memory
  Budget=78 (ratio=0.2) | Recent window=39
  Match: 14/200 (7.0%)
  Baseline:  In this paper, we propose a novel approach to address the memory bottleneck of large language models. We propose a novel approach to
  H2O:       In this paper, we propose a novel approach to offload the computation of the transformer to the GPU

In [ ]:
long_unique = """Large Language Models (LLMs), despite their recent impressive accomplishments,
are notably cost-prohibitive to deploy, particularly for applications involving long-
content generation, such as dialogue systems and story writing. Often, a large
amount of transient state information, referred to as the KV cache, is stored in GPU
memory in addition to model parameters, scaling linearly with the sequence length
and batch size. In this paper, we introduce a novel approach for implementing the
KV cache which significantly reduces its memory footprint. Our approach is based
on the noteworthy observation that a small portion of tokens contributes most of
the value when computing attention scores. We call these tokens Heavy Hitters
(H2). Through a comprehensive investigation, we find that (i) the emergence of H2
is natural and strongly correlates with the frequent co-occurrence of tokens in the
text, and (ii) removing them results in significant performance degradation. Based
on these insights, we propose Heavy Hitter Oracle (H2O), a KV cache eviction
policy that dynamically retains a balance of recent and H2 tokens. We formulate
the KV cache eviction as a dynamic submodular problem and prove (under mild
assumptions) a theoretical guarantee for our novel eviction algorithm which could
help guide future work. We validate the accuracy of our algorithm with OPT,
LLaMA, and GPT-NeoX across a wide range of tasks. Our implementation of
H2O with 20% heavy hitters improves the throughput over three leading inference
systems DeepSpeed Zero-Inference, Hugging Face Accelerate, and FlexGen by
up to 29×, 29×, and 3×on OPT-6.7B and OPT-30B. With the same batch size,
H2O can reduce the latency by up to 1.9×. The code is available at https:
//github.com/FMInference/H2O."""

validate_h2o_proper(long_unique, max_new_tokens=200, budget_ratio=0.6)
validate_h2o_proper(long_unique, max_new_tokens=200, budget_ratio=0.3)
validate_h2o_proper(long_unique, max_new_tokens=200, budget_ratio=0.2)

  Budget=364 (ratio=0.6) | Recent window=182
  Match: 200/200 (100.0%)
  Baseline: </s>I'm not sure if this is a good thing or a bad thing.
It's a good thing.  
  H2O:      </s>I'm not sure if this is a good thing or a bad thing.
It's a good thing.  
  Budget=182 (ratio=0.3) | Recent window=91
  Match: 200/200 (100.0%)
  Baseline: </s>I'm not sure if this is a good thing or a bad thing.
It's a good thing.  
  H2O:      </s>I'm not sure if this is a good thing or a bad thing.
It's a good thing.  
  Budget=121 (ratio=0.2) | Recent window=60
  Match: 17/200 (8.5%)
  Baseline: </s>I'm not sure if this is a good thing or a bad thing.
It's a good thing.  
  H2O:      </s>I'm not sure if this is a good thing or a bad thing.
I'm not sure if this is a


In [ ]:
long_unique = """Large language models have revolutionized natural language processing by demonstrating remarkable capabilities in text generation, translation, summarization, and question answering. These models, built on the transformer architecture introduced by Vaswani et al in 2017, use self-attention mechanisms to capture long-range dependencies in text. The key innovation of transformers is the multi-head attention mechanism, which allows the model to attend to information from different representation subspaces at different positions. During inference, the model generates tokens autoregressively, producing one token at a time based on the entire previous context. This requires storing key and value tensors from all previous tokens in what is known as the KV cache. As sequence lengths grow, the KV cache becomes a significant memory bottleneck, consuming gigabytes of GPU memory for long contexts. Several approaches have been proposed to address this challenge, including quantization of cached values, eviction of less important entries, and offloading to CPU memory."""

validate_h2o_proper(long_unique, max_new_tokens=200, budget_ratio=0.5)
validate_h2o_proper(long_unique, max_new_tokens=200, budget_ratio=0.3)
validate_h2o_proper(long_unique, max_new_tokens=200, budget_ratio=0.2)

  Budget=197 (ratio=0.5) | Recent window=98
  Match: 12/200 (6.0%)
  Baseline:  In this paper, we propose a novel approach to address the memory bottleneck of large language models. We propose a novel approach to
  H2O:       In this paper, we propose a novel approach to address this problem, called the KV-cache-free model. The
  Budget=118 (ratio=0.3) | Recent window=59
  Match: 13/200 (6.5%)
  Baseline:  In this paper, we propose a novel approach to address the memory bottleneck of large language models. We propose a novel approach to
  H2O:       In this paper, we propose a novel approach to offload the memory burden of large language models. We use a novel memory
  Budget=78 (ratio=0.2) | Recent window=39
  Match: 14/200 (7.0%)
  Baseline:  In this paper, we propose a novel approach to address the memory bottleneck of large language models. We propose a novel approach to
  H2O:       In this paper, we propose a novel approach to offload the computation of the transformer to the GPU

In [ ]:
validate_h2o_proper(long_unique, max_new_tokens=200, budget_ratio=0.6)

  Budget=236 (ratio=0.6) | Recent window=118
  Match: 200/200 (100.0%)
  Baseline:  In this paper, we propose a novel approach to address the memory bottleneck of large language models. We propose a novel approach to
  H2O:       In this paper, we propose a novel approach to address the memory bottleneck of large language models. We propose a novel approach to


# **H2O Perplexity eval**

In [ ]:
def evaluate_perplexity_h2o(text, budget_ratio=1.0, recent_ratio=0.5, stride=512):
    """Compute perplexity with H2O eviction at given budget ratio. 1.0 = no eviction (baseline)."""
    encodings = tokenizer(text, return_tensors="pt")
    input_ids = encodings.input_ids.to(device)
    seq_len = input_ids.shape[1]

    nlls = []

    with torch.no_grad():
        # Process full sequence token by token
        kv = DynamicCache()
        cum_scores = None

        for i in range(0, seq_len - 1):
            token = input_ids[:, i:i+1]

            outputs = model(
                input_ids=token,
                past_key_values=kv,
                use_cache=True,
                output_attentions=True
            )
            kv = outputs.past_key_values

            # Compute NLL for next token prediction
            logits = outputs.logits[:, -1, :]
            target = input_ids[:, i + 1]
            nll = torch.nn.functional.cross_entropy(logits, target)
            nlls.append(nll.item())

            # Update cumulative scores
            current_len = kv.get_seq_length()
            step_scores = torch.zeros(current_len, device=device)
            for attn in outputs.attentions:
                scores = attn[0].sum(dim=0).sum(dim=0)
                if len(scores) == current_len:
                    step_scores += scores

            if cum_scores is None:
                cum_scores = step_scores
            else:
                new_score = torch.zeros(1, device=device)
                cum_scores = torch.cat([cum_scores, new_score])
                if len(cum_scores) == current_len:
                    cum_scores += step_scores

            # H2O eviction if over budget
            if budget_ratio < 1.0:
                budget = max(2, int(seq_len * budget_ratio))
                recent_window = int(budget * recent_ratio)

                while current_len > budget:
                    evict_candidates = cum_scores[:-recent_window].clone()
                    if len(evict_candidates) == 0:
                        break

                    evict_pos = evict_candidates.argmin().item()

                    keep_mask = torch.ones(current_len, dtype=torch.bool, device=device)
                    keep_mask[evict_pos] = False
                    keep_indices = torch.where(keep_mask)[0]

                    kv_layers = unpack_kv_cache(kv)
                    kept_layers = [(k[:, :, keep_indices, :], v[:, :, keep_indices, :]) for k, v in kv_layers]
                    kv = build_kv_cache(kept_layers)

                    cum_scores = cum_scores[keep_mask]
                    current_len -= 1

            del outputs

    avg_nll = sum(nlls) / len(nlls)
    perplexity = torch.exp(torch.tensor(avg_nll)).item()
    return perplexity, len(nlls)


# Get evaluation text — use a real paragraph
eval_text = """The transformer architecture was introduced in 2017 by Vaswani et al. in their landmark paper Attention Is All You Need. The architecture relies entirely on self-attention mechanisms, dispensing with recurrence and convolutions entirely. Since its introduction, the transformer has become the foundation for nearly all modern large language models including GPT, BERT, T5, LLaMA, and many others. The key innovation was the multi-head self-attention mechanism which allows the model to attend to different positions in the input sequence simultaneously. This parallel attention computation made transformers significantly more efficient to train than recurrent neural networks which process tokens sequentially. The original transformer was designed for machine translation but researchers quickly discovered that the architecture could be adapted for virtually any natural language processing task. Decoder-only variants like GPT use causal attention masks to enable autoregressive generation where each token can only attend to previous tokens. During inference these models store key and value tensors from all previous tokens in what is known as the KV cache to avoid redundant computation. As sequence lengths and batch sizes grow the KV cache becomes a dominant consumer of GPU memory often exceeding the memory required for model parameters themselves. This has motivated extensive research into KV cache compression including quantization eviction policies and offloading strategies."""

print("Evaluating perplexity across different H2O budget ratios...")
print(f"Text length: {len(tokenizer.encode(eval_text))} tokens\n")

for ratio in [1.0, 0.8, 0.6, 0.5, 0.4, 0.3, 0.2]:
    label = "Full (baseline)" if ratio == 1.0 else f"H2O {int(ratio*100)}% budget"
    ppl, n_tokens = evaluate_perplexity_h2o(eval_text, budget_ratio=ratio)
    print(f"  {label:<25} | Perplexity: {ppl:.2f} | Tokens: {n_tokens}")

Evaluating perplexity across different H2O budget ratios...
Text length: 260 tokens

  Full (baseline)           | Perplexity: 23.84 | Tokens: 259
  H2O 80% budget            | Perplexity: 23.85 | Tokens: 259
  H2O 60% budget            | Perplexity: 23.50 | Tokens: 259
  H2O 50% budget            | Perplexity: 23.51 | Tokens: 259
  H2O 40% budget            | Perplexity: 23.58 | Tokens: 259
  H2O 30% budget            | Perplexity: 24.48 | Tokens: 259
  H2O 20% budget            | Perplexity: 26.36 | Tokens: 259


# **Extra**

In [ ]:
def run_h2o_dejavu_prefetch_pinned_benchmark(budget_ratio=0.4, recent_ratio=0.5):
    microbatches_count = [4, 8, 16]
    n_tokens = [128, 256, 512]
    results = []

    total_vram_mb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 2)
    num_layers = model.config.num_hidden_layers

    for D in microbatches_count:
        for N in n_tokens:

            prompts = LONG_PROMPTS[:D]
            torch.cuda.empty_cache()
            torch.cuda.reset_peak_memory_stats()

            # Compute budget per microbatch
            sample_len = tokenizer(prompts[0], return_tensors="pt").input_ids.shape[1]
            budget = int((sample_len + N) * budget_ratio)
            budget = max(budget, sample_len)
            recent_window = int(budget * recent_ratio)

            total_start = time.perf_counter()

            with torch.no_grad():
                next_tokens = []
                cpu_kv_caches = []
                cum_scores_list = []

                # PREFILL
                for i in range(D):
                    inputs = tokenizer(prompts[i], return_tensors="pt").to(device)
                    outputs = model(
                        input_ids=inputs.input_ids,
                        use_cache=True,
                        output_attentions=True
                    )
                    next_token = torch.argmax(outputs.logits[:, -1, :], dim=1).unsqueeze(1)
                    next_tokens.append(next_token)

                    # Initialize cumulative scores
                    seq_len = outputs.past_key_values.get_seq_length()
                    scores = torch.zeros(seq_len, device=device)
                    for attn in outputs.attentions:
                        scores += attn[0].sum(dim=0).sum(dim=0)
                    cum_scores_list.append(scores)

                    # Offload to pinned CPU memory
                    cpu_kv_caches.append(offload_kv_to_cpu(outputs.past_key_values))

                prefetch_stream = torch.cuda.Stream()
                offload_stream = torch.cuda.Stream()

                # Prefetch first microbatch
                with torch.cuda.stream(prefetch_stream):
                    prefetched = tuple(
                        (k.to(device, non_blocking=True), v.to(device, non_blocking=True))
                        for k, v in cpu_kv_caches[0]
                    )

                # DECODE
                for step in range(N - 1):
                    for i in range(D):
                        next_i = (i + 1) % D

                        prefetch_stream.synchronize()
                        offload_stream.synchronize()

                        gpu_layers = prefetched

                        # Prefetch next microbatch
                        with torch.cuda.stream(prefetch_stream):
                            prefetched = tuple(
                                (k.to(device, non_blocking=True), v.to(device, non_blocking=True))
                                for k, v in cpu_kv_caches[next_i]
                            )

                        # Build cache and decode
                        gpu_cache = build_kv_cache(gpu_layers)
                        outputs = model(
                            input_ids=next_tokens[i],
                            past_key_values=gpu_cache,
                            use_cache=True,
                            output_attentions=True
                        )
                        next_tokens[i] = torch.argmax(outputs.logits[:, -1, :], dim=1).unsqueeze(1)

                        # Update cumulative scores
                        kv = outputs.past_key_values
                        current_len = kv.get_seq_length()
                        new_score = torch.zeros(1, device=device)
                        cum_scores_list[i] = torch.cat([cum_scores_list[i], new_score])
                        for attn in outputs.attentions:
                            step_scores = attn[0].sum(dim=0).sum(dim=0)
                            if len(step_scores) == len(cum_scores_list[i]):
                                cum_scores_list[i] += step_scores

                        # H2O eviction — one at a time
                        while current_len > budget:
                            evict_candidates = cum_scores_list[i][:-recent_window].clone()
                            if len(evict_candidates) == 0:
                                break

                            evict_pos = evict_candidates.argmin().item()

                            keep_mask = torch.ones(current_len, dtype=torch.bool, device=device)
                            keep_mask[evict_pos] = False
                            keep_indices = torch.where(keep_mask)[0]

                            kv_layers = unpack_kv_cache(kv)
                            kept_layers = [(k[:, :, keep_indices, :], v[:, :, keep_indices, :]) for k, v in kv_layers]
                            kv = build_kv_cache(kept_layers)

                            cum_scores_list[i] = cum_scores_list[i][keep_mask]
                            current_len -= 1

                        # Offload evicted cache to pinned CPU
                        with torch.cuda.stream(offload_stream):
                            cpu_kv_caches[i] = tuple(
                                (k.detach().contiguous().cpu().pin_memory(),
                                 v.detach().contiguous().cpu().pin_memory())
                                for k, v in unpack_kv_cache(kv)
                            )

                        del gpu_layers, gpu_cache, outputs, kv

            total_end = time.perf_counter()

            latency = total_end - total_start
            throughput = (D * N) / latency
            peak_vram_mb = torch.cuda.max_memory_allocated() / (1024 ** 2)
            vram_util = (peak_vram_mb / total_vram_mb) * 100

            cpu_kv_mb = sum(
                k.nelement() * k.element_size() + v.nelement() * v.element_size()
                for cache in cpu_kv_caches for k, v in cache
            ) / (1024 ** 2)

            print(f"D={D:<4} N={N:<4} | Budget={budget:<4} | Latency={latency:.3f}s | Throughput={throughput:.1f} tok/s | "
                  f"Peak VRAM={peak_vram_mb:.1f}MB | VRAM%={vram_util:.1f}% | CPU KV={cpu_kv_mb:.1f}MB")

            results.append({
                "Microbatches (D)": D,
                "Tokens Generated (N)": N,
                "Budget": budget,
                "Latency (s)": latency,
                "Throughput (tok/s)": throughput,
                "Peak VRAM (MB)": peak_vram_mb,
                "VRAM Util (%)": vram_util,
                "CPU KV (MB)": cpu_kv_mb,
            })

            del cpu_kv_caches, next_tokens, cum_scores_list, prefetched
            torch.cuda.empty_cache()

    return pd.DataFrame(results)

In [ ]:
run_h2o_dejavu_prefetch_pinned_benchmark()

D=4    N=128  | Budget=74   | Latency=25.926s | Throughput=19.7 tok/s | Peak VRAM=12896.0MB | VRAM%=31.9% | CPU KV=148.0MB
D=4    N=256  | Budget=125  | Latency=56.783s | Throughput=18.0 tok/s | Peak VRAM=13023.9MB | VRAM%=32.2% | CPU KV=250.0MB
D=4    N=512  | Budget=228  | Latency=130.649s | Throughput=15.7 tok/s | Peak VRAM=13294.1MB | VRAM%=32.9% | CPU KV=456.0MB
D=8    N=128  | Budget=74   | Latency=51.110s | Throughput=20.0 tok/s | Peak VRAM=13080.1MB | VRAM%=32.3% | CPU KV=296.0MB
D=8    N=256  | Budget=125  | Latency=113.365s | Throughput=18.1 tok/s | Peak VRAM=13024.0MB | VRAM%=32.2% | CPU KV=500.0MB
D=8    N=512  | Budget=228  | Latency=264.235s | Throughput=15.5 tok/s | Peak VRAM=13294.4MB | VRAM%=32.9% | CPU KV=912.0MB
D=16   N=128  | Budget=74   | Latency=108.898s | Throughput=18.8 tok/s | Peak VRAM=13074.6MB | VRAM%=32.3% | CPU KV=592.0MB
D=16   N=256  | Budget=125  | Latency=236.131s | Throughput=17.3 tok/s | Peak VRAM=13024.0MB | VRAM%=32.2% | CPU KV=1000.0MB
D=16   N=5

,Microbatches (D),Tokens Generated (N),Budget,Latency (s),Throughput (tok/s),Peak VRAM (MB),VRAM Util (%),CPU KV (MB)
0,4,128,74,25.925889,19.748599,12895.957520,31.888029,148.0
1,4,256,125,56.782836,18.033618,13023.949707,32.204518,250.0
2,4,512,228,130.649469,15.675533,13294.141113,32.872624,456.0
3,8,128,74,51.110460,20.035038,13080.143066,32.343468,296.0
4,8,256,125,113.364562,18.065610,13023.952637,32.204525,500.0
5,8,512,228,264.235304,15.501335,13294.388184,32.873235,912.0
6,16,128,74,108.897896,18.806608,13074.562500,32.329669,592.0
7,16,256,125,236.131009,17.346303,13023.960449,32.204544,1000.0
8,16,512,228,529.534955,15.470178,13294.509277,32.873534,1824.0


In [ ]:
def run_h2o_gpu_only_benchmark(budget_ratio=0.4, recent_ratio=0.5):
    """H2O eviction on GPU only — no CPU offloading, no prefetching."""
    microbatches_count = [4, 8, 16]
    n_tokens = [128, 256, 512]
    results = []

    total_vram_mb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 2)

    for D in microbatches_count:
        for N in n_tokens:

            prompts = LONG_PROMPTS[:D]
            torch.cuda.empty_cache()
            torch.cuda.reset_peak_memory_stats()

            sample_len = tokenizer(prompts[0], return_tensors="pt").input_ids.shape[1]
            budget = int((sample_len + N) * budget_ratio)
            budget = max(budget, sample_len)
            recent_window = int(budget * recent_ratio)

            total_start = time.perf_counter()

            with torch.no_grad():
                kv_caches = []
                next_tokens = []
                cum_scores_list = []

                # PREFILL
                for i in range(D):
                    inputs = tokenizer(prompts[i], return_tensors="pt").to(device)
                    outputs = model(
                        input_ids=inputs.input_ids,
                        use_cache=True,
                        output_attentions=True
                    )
                    next_token = torch.argmax(outputs.logits[:, -1, :], dim=1).unsqueeze(1)
                    next_tokens.append(next_token)

                    seq_len = outputs.past_key_values.get_seq_length()
                    scores = torch.zeros(seq_len, device=device)
                    for attn in outputs.attentions:
                        scores += attn[0].sum(dim=0).sum(dim=0)
                    cum_scores_list.append(scores)
                    kv_caches.append(outputs.past_key_values)

                # DECODE
                for step in range(N - 1):
                    for i in range(D):

                        outputs = model(
                            input_ids=next_tokens[i],
                            past_key_values=kv_caches[i],
                            use_cache=True,
                            output_attentions=True
                        )
                        kv_caches[i] = outputs.past_key_values
                        next_tokens[i] = torch.argmax(outputs.logits[:, -1, :], dim=1).unsqueeze(1)

                        # Update cumulative scores
                        current_len = kv_caches[i].get_seq_length()
                        new_score = torch.zeros(1, device=device)
                        cum_scores_list[i] = torch.cat([cum_scores_list[i], new_score])
                        for attn in outputs.attentions:
                            step_scores = attn[0].sum(dim=0).sum(dim=0)
                            if len(step_scores) == len(cum_scores_list[i]):
                                cum_scores_list[i] += step_scores

                        # H2O eviction
                        while current_len > budget:
                            evict_candidates = cum_scores_list[i][:-recent_window].clone()
                            if len(evict_candidates) == 0:
                                break

                            evict_pos = evict_candidates.argmin().item()

                            keep_mask = torch.ones(current_len, dtype=torch.bool, device=device)
                            keep_mask[evict_pos] = False
                            keep_indices = torch.where(keep_mask)[0]

                            kv_layers = unpack_kv_cache(kv_caches[i])
                            kept_layers = [(k[:, :, keep_indices, :], v[:, :, keep_indices, :]) for k, v in kv_layers]
                            kv_caches[i] = build_kv_cache(kept_layers)

                            cum_scores_list[i] = cum_scores_list[i][keep_mask]
                            current_len -= 1

                        del outputs

            total_end = time.perf_counter()

            latency = total_end - total_start
            throughput = (D * N) / latency
            peak_vram_mb = torch.cuda.max_memory_allocated() / (1024 ** 2)
            vram_util = (peak_vram_mb / total_vram_mb) * 100

            kv_bytes = 0
            for cache in kv_caches:
                kv_layers = unpack_kv_cache(cache)
                for k, v in kv_layers:
                    kv_bytes += k.nelement() * k.element_size()
                    kv_bytes += v.nelement() * v.element_size()
            kv_mb = kv_bytes / (1024 ** 2)

            print(f"D={D:<4} N={N:<4} | Budget={budget:<4} | Latency={latency:.3f}s | Throughput={throughput:.1f} tok/s | "
                  f"Peak VRAM={peak_vram_mb:.1f}MB | VRAM%={vram_util:.1f}% | GPU KV={kv_mb:.1f}MB")

            results.append({
                "Microbatches (D)": D,
                "Tokens Generated (N)": N,
                "Budget": budget,
                "Latency (s)": latency,
                "Throughput (tok/s)": throughput,
                "Peak VRAM (MB)": peak_vram_mb,
                "VRAM Util (%)": vram_util,
                "GPU KV (MB)": kv_mb,
            })

            del kv_caches, next_tokens, cum_scores_list
            torch.cuda.empty_cache()

    return pd.DataFrame(results)

In [ ]:
print("=" * 80)
print("H2O GPU-ONLY (40% budget)")
print("=" * 80)
h2o_gpu_results = run_h2o_gpu_only_benchmark(budget_ratio=0.4)

print("\n" + "=" * 80)
print("H2O + DEJAVU + PREFETCH + PINNED (40% budget)")
print("=" * 80)
h2o_dejavu_results = run_h2o_dejavu_prefetch_pinned_benchmark(budget_ratio=0.4)

H2O GPU-ONLY (40% budget)
D=4    N=128  | Budget=74   | Latency=16.975s | Throughput=30.2 tok/s | Peak VRAM=12933.0MB | VRAM%=32.0% | GPU KV=148.0MB
D=4    N=256  | Budget=125  | Latency=33.405s | Throughput=30.7 tok/s | Peak VRAM=13123.4MB | VRAM%=32.5% | GPU KV=250.0MB
D=4    N=512  | Budget=228  | Latency=66.430s | Throughput=30.8 tok/s | Peak VRAM=13473.4MB | VRAM%=33.3% | GPU KV=456.0MB
D=8    N=128  | Budget=74   | Latency=33.804s | Throughput=30.3 tok/s | Peak VRAM=13249.6MB | VRAM%=32.8% | GPU KV=296.0MB
D=8    N=256  | Budget=125  | Latency=66.486s | Throughput=30.8 tok/s | Peak VRAM=13373.5MB | VRAM%=33.1% | GPU KV=500.0MB
D=8    N=512  | Budget=228  | Latency=131.666s | Throughput=31.1 tok/s | Peak VRAM=13939.8MB | VRAM%=34.5% | GPU KV=912.0MB
D=16   N=128  | Budget=74   | Latency=67.637s | Throughput=30.3 tok/s | Peak VRAM=13513.2MB | VRAM%=33.4% | GPU KV=592.0MB
D=16   N=256  | Budget=125  | Latency=132.916s | Throughput=30.8 tok/s | Peak VRAM=13873.5MB | VRAM%=34.3% | GPU

KeyboardInterrupt: 